# Building a comprehensive RAG system

This notebook is a **step-by-step lab** for building **Retrieval-Augmented Generation (RAG)** from a single regulatory PDF. You will go from raw pages to embeddings, vector search, grounded prompting, and qualitative checks for hallucination risk.

## Document and runtime

- **Corpus**: `data/IGF Code (124Pages).pdf`
- **Primary LLM path (Section A)**: local inference with **`llama-cpp-python`** and a **GGUF** model
- **Optional service LLM (Section B)**: call **Ollama** through `ollama_model_runner.py` (RAG logic stays in the notebook)
- **Framework tracks (optional)**: **LangChain** (Section C) and **LlamaIndex** (Section D) reuse the same `manual_chunks` for fair comparison

## What you will build

1. **Ingestion**: extract text per page from the PDF  
2. **Chunking**: compare chunking strategies and pick a default for the pipeline  
3. **Embeddings**: encode chunks with Sentence Transformers; sanity-check vectors  
4. **Indexing**: FAISS (core) and Chroma (optional persistence)  
5. **Retrieval**: top‑k search, context formatting, and simple experiments  
6. **Generation**: grounded prompts and local completion  
7. **Trust and quality**: citations, light validation, and small multi-question checks  
8. **Extensions**: Ollama client workflow; LangChain and LlamaIndex retrieval patterns

## How to use this notebook

- Run **Section A** steps in order from **Step 0** through the end-to-end RAG cells unless your instructor says otherwise.  
- **Section B** is optional and assumes Ollama is installed and running separately.  
- **Sections C and D** are optional add-ons; complete chunking in Section A first so `manual_chunks` exists.  
- If installs change your environment, **restart the kernel** once, then rerun from Step 0.

## Prerequisites (conceptual)

Comfort with Python in Jupyter, basic idea of vectors and cosine similarity, and patience for first-time model downloads.

> **Tip:** Keep the retrieved context visible when reading model answers — most “RAG bugs” are actually retrieval or chunking issues, not the LLM alone.

## Section A - Notebook-based RAG (without Ollama)

This section runs the full RAG workflow directly in notebook cells using `llama-cpp-python` for local generation.

## Step 0 — `llama-cpp-python` + GGUF setup for Colab

This notebook uses **in-process local inference** with [`llama-cpp-python`](https://github.com/abetlen/llama-cpp-python) and a **`.gguf`** model file.

### What this step does

1. Install/verify required packages in the current notebook kernel.
2. Download a default GGUF model from Hugging Face (optional).
3. Set `LLAMA_CPP_GGUF` so later cells can load the model.

### Required packages

- `llama-cpp-python`
- `huggingface_hub`
- `pypdf`
- `sentence-transformers`
- `faiss-cpu`
- `chromadb` (optional)

### Notes for Colab

- First model download can be large and take time.
- If using a gated model, set `HF_TOKEN` first.
- If you already have a GGUF file path, set `LLAMA_CPP_GGUF` and skip the download cell.
- If import errors remain after install, restart runtime and rerun from Step 0a.

### Notes for conda / local Linux

- If `llama-cpp-python` tries to compile and CMake reports a bad `CC` (often `gcc -pthread -B .../compiler_compat`), Step 0a clears `CC`/`CXX` for that install and prefers wheels from the maintainer index so a local compiler is usually not required.

In [ ]:
!python -m pip install --upgrade pip

In [2]:
# Step 0a - Install dependencies in THIS notebook kernel (Colab-friendly)
#
# Conda can set CC/CXX to "gcc -pthread -B .../compiler_compat", which breaks CMake inside
# pip's build isolation. Clear them for this install only, and prefer pre-built wheels.
import os
import platform
import subprocess
import sys

_install_env = os.environ.copy()
for _k in ("CC", "CXX"):
    _install_env.pop(_k, None)

_extra = "https://abetlen.github.io/llama-cpp-python/whl/cpu"
if platform.system() == "Darwin" and platform.machine() == "arm64":
    _extra = "https://abetlen.github.io/llama-cpp-python/whl/metal"

subprocess.check_call(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "--prefer-binary",
        "--extra-index-url",
        _extra,
        "llama-cpp-python",
        "huggingface_hub",
        "pypdf",
        "sentence-transformers",
        "faiss-cpu",
        "chromadb",
    ],
    env=_install_env,
)

0

In [3]:
# Step 0a.1 - Verify critical imports
import llama_cpp  # noqa: F401
import huggingface_hub  # noqa: F401

print("OK - llama-cpp-python and huggingface_hub are ready in this kernel.")
print("Next: run Step 0b to download GGUF (or set LLAMA_CPP_GGUF manually).")

OK - llama-cpp-python and huggingface_hub are ready in this kernel.
Next: run Step 0b to download GGUF (or set LLAMA_CPP_GGUF manually).


In [4]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [5]:
# Step 0a.2 - Import shared libraries used in later steps
import os
import re
import json
from pathlib import Path
from typing import List, Dict

import numpy as np
import pandas as pd
from pypdf import PdfReader
from huggingface_hub import hf_hub_download
from sentence_transformers import SentenceTransformer
import faiss
from llama_cpp import Llama

try:
    import chromadb
    from chromadb.config import Settings
    CHROMA_AVAILABLE = True
except Exception:
    CHROMA_AVAILABLE = False

print('Shared imports ready. Chroma available:', CHROMA_AVAILABLE)

/home/ethan/anaconda3/envs/kmouc/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Shared imports ready. Chroma available: True


In [6]:
# Step 0b — Download default GGUF from Hugging Face (large download on first run)
import os

from huggingface_hub import hf_hub_download

# Defaults: small instruct chat model, Q4_K_M — override via env before running this cell
HF_GGUF_REPO = os.environ.get(
    "HF_GGUF_REPO", "TheBloke/TinyLlama-1.1B-Chat-v1.0-GGUF"
).strip()
HF_GGUF_FILE = os.environ.get(
    "HF_GGUF_FILE", "tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf"
).strip()
_HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")

_existing = (os.environ.get("LLAMA_CPP_GGUF") or "").strip()
if _existing and os.path.isfile(_existing):
    print("Using existing LLAMA_CPP_GGUF — skip download:", _existing)
else:
    print(f"Downloading from Hub: {HF_GGUF_REPO} / {HF_GGUF_FILE} ...")
    path = hf_hub_download(
        repo_id=HF_GGUF_REPO, filename=HF_GGUF_FILE, token=_HF_TOKEN
    )
    os.environ["LLAMA_CPP_GGUF"] = path
    print("Set LLAMA_CPP_GGUF to:", path)

print("Done. Continue to run next Steps")


Set LLAMA_CPP_GGUF to: /home/ethan/.cache/huggingface/hub/models--TheBloke--TinyLlama-1.1B-Chat-v1.0-GGUF/snapshots/52e7645ba7c309695bec7ac98f4f005b139cf465/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf
Done. Continue to run next Steps


## Step 1 - Load source data from PDF (`IGF Code`)

In this step, we convert the raw PDF into a structured in-memory dataset that can be chunked and indexed later.

Main source file:
- `data/IGF Code (124Pages).pdf`

### Why this step matters
- RAG quality starts with ingestion quality.
- If extraction is noisy or missing sections, downstream retrieval and generation will degrade.

### What the code does
1. Loads each PDF page.
2. Extracts text and normalizes whitespace.
3. Discards very short/empty pages.
4. Builds a record with `page`, `title`, and `text`.

### Expected output
- Number of loaded pages.
- A short preview from one page to verify extraction quality.

If the preview looks broken (garbled text, many missing lines), fix extraction first before continuing.

In [7]:
# Optional quick check: confirm model path for later Step 7
print('LLAMA_CPP_GGUF =', os.environ.get('LLAMA_CPP_GGUF', '(not set)'))

LLAMA_CPP_GGUF = /home/ethan/.cache/huggingface/hub/models--TheBloke--TinyLlama-1.1B-Chat-v1.0-GGUF/snapshots/52e7645ba7c309695bec7ac98f4f005b139cf465/tinyllama-1.1b-chat-v1.0.Q4_K_M.gguf


### Step 1 - Read PDF pages into records and execute PDF extraction

Before chunking, extract page text and store structured records (`page`, `title`, `text`).
This workshop uses **only these PDF-derived records** as the retrieval corpus.

Run the next cell to parse all pages from the IGF PDF and create `manual_docs`.
After this, we will start chunking strategies.

In [8]:
pdf_path = Path('data') / 'IGF Code (124Pages).pdf'
if not pdf_path.exists():
    raise FileNotFoundError(f'PDF not found: {pdf_path}. Please verify the file path.')

reader = PdfReader(str(pdf_path))
manual_docs = []
for i, page in enumerate(reader.pages, start=1):
    text = page.extract_text() or ''
    text = re.sub(r'\s+', ' ', text).strip()
    if len(text) < 40:
        continue
    manual_docs.append({
        'product_id': 'IGF-CODE',
        'title': f'IGF Code - Page {i}',
        'page': i,
        'text': text,
    })

print('Loaded pages from PDF:', len(manual_docs))
if manual_docs:
    print('Sample page title:', manual_docs[0]['title'])
    print('Sample text preview:', manual_docs[0]['text'][:300])

Loaded pages from PDF: 124
Sample page title: IGF Code - Page 1
Sample text preview: MSC 95/22/Add.1 Annex 1, page 1 I:\MSC\95\MSC 95-22-ADD.1 (E).docx ANNEX 1 RESOLUTION MSC.391(95) (adopted on 11 June 2015) ADOPTION OF THE INTERNATIONAL CODE OF SAFETY FOR SHIPS USING GASES OR OTHER LOW-FLASHPOINT FUELS (IGF CODE) THE MARITIME SAFETY COMMITTEE, RECALLING Article 28(b) of the Conven


## Step 2 - Expanded chunking strategies (compare multiple methods)

Chunking strongly affects retrieval quality. In this step, we compare several methods on IGF PDF text:

1. **Sentence-window semantic chunking** (default): sentence-based windows + overlap.
2. **Fixed-size word chunking**: deterministic and simple baseline.
3. **Heading-aware chunking**: split by detected section-like headings, then window.

Why compare?
- Smaller chunks improve precision but may lose context.
- Larger chunks keep context but may reduce retrieval specificity.
- Heading-aware chunks often improve legal/regulatory document retrieval.

At the end of this step, we choose one method as the default for downstream embedding/indexing.

### Step 2.1 — Define chunking functions / Định nghĩa các hàm chunking

Ô code bên dưới gom toàn bộ **logic cắt văn bản (chunking)** thành các hàm Python tái sử dụng cho các bước 2.2 trở đi. Mục tiêu: biến **một trang PDF đã trích xuất** thành **nhiều đoạn nhỏ** phù hợp để nhúng (embedding), tìm kiếm vector và đưa vào prompt RAG.

#### Luồng xử lý tổng thể (để hình dung trước khi đọc code)

1. **Đầu vào**: chuỗi `text` của từng trang (đã làm sạch khoảng trắng ở bước trước).
2. **Tách câu hoặc từ / tiêu đề**: tùy phương pháp, ta tách theo ranh giới câu, theo số từ cố định, hoặc theo dòng giống mục lục.
3. **Gom thành chunk**: mỗi chunk là một đoạn văn bản ngắn hơn cả trang nhưng vẫn giữ đủ ngữ cảnh cho câu hỏi người dùng.
4. **Chuẩn hóa đầu ra**: hàm `build_manual_chunks` gói mỗi chunk vào một **dictionary** thống nhất (có `chunk_id`, số trang, phương pháp, v.v.) để các bước sau không phải quan tâm chi tiết từng kiểu chunking.

---

#### `sentence_split(text)`

- **Chức năng**: tách một đoạn văn dài thành **danh sách các câu** (string).
- **Cách làm**: `re.sub(r'\s+', ' ', ...)` gộp mọi khoảng trắng (xuống dòng, tab, nhiều space) thành một space; sau đó `re.split(r'(?<=[.!?])\s+', ...)` cắt **sau dấu `.` `!` `?`** rồi theo khoảng trắng — tức là ranh giới câu kiểu “nhẹ”, không cần thư viện NLP nặng.
- **Hạn chế cần biết**: tài liệu kỹ thuật có `Fig. 2.3` hay `Dr.` có thể bị tách “sai câu” so với người; đây là đánh đổi để code đơn giản, dễ đọc trong khóa học.

---

#### `chunk_sentence_window(text, max_words, overlap_sentences)`

- **Chức năng**: gom **nhiều câu liên tiếp** vào một chunk, sao cho tổng số **từ** (đếm bằng `len(câu.split())`) không vượt `max_words`; khi đầy thì **đóng chunk** và mở chunk mới.
- **Overlap (chồng lấn)**: `overlap_sentences` là số **câu cuối** của chunk hiện tại được **copy sang đầu** chunk kế tiếp. Việc này giúp ngữ cảnh ở biên hai chunk không bị “đứt đoạn” hoàn toàn khi embedding hoặc khi truy vấn khớp biên câu.
- **Vòng lặp trong code**: biến `current` giữ các câu đang gom; `current_len` đếm từ; khi thêm một câu mới làm vượt `max_words`, chunk cũ được `append`, rồi `current` được reset bằng phần đuôi overlap + câu mới.

---

#### `chunk_fixed_words(text, chunk_words, overlap_words)`

- **Chức năng**: cắt theo **số từ cố định** — baseline đơn giản, luôn cho kết quả dễ lặp lại (deterministic).
- **`step`**: `chunk_words - overlap_words` (tối thiểu 1) là bước trượt cửa sổ: mỗi lần lấy `chunk_words` từ, rồi nhảy `step` từ về phía sau để chunk kế có phần **chồng lấn** `overlap_words` từ với chunk trước (khi overlap < chunk_words).
- **Ưu / nhược**: rất dễ hiểu và so sánh tham số; nhưng có thể **cắt giữa câu** hoặc giữa mệnh đề pháp lý, nên chất lượng retrieval đôi khi kém hơn chunk theo câu hoặc theo tiêu đề.

---

#### `chunk_heading_aware(text, max_words)`

- **Chức năng**: cố gắng **tôn trọng cấu trúc** văn bản quy phạm: tách theo dòng/câu, nhận dòng bắt đầu giống **tiêu đề mục** (Chapter, Section, Part, hoặc dạng số `1.2.3`), rồi với mỗi khối “mục” lại dùng `chunk_fixed_words` với `chunk_words=max_words` để không có mục quá dài.
- **`heading_pattern`**: biểu thức chính quy (regex) nhận diện dòng có vẻ là đầu mục; khi gặp mục mới và đã có nội dung trong `current`, toàn bộ `current` được đẩy thành một **section** rồi bắt đầu section mới từ dòng tiêu đề đó.
- **Lưu ý**: đây là heuristic (quy tắc gần đúng), không phải phân tích cấu trúc PDF thật; vẫn hữu ích để học viên thấy ý tưởng “chunk theo cấu trúc”.

---

#### `build_manual_chunks(docs, method, params)`

- **Chức năng**: **một cửa** gọi đúng một trong ba chiến lược (`sentence_window`, `fixed_words`, `heading_aware`) cho **toàn bộ** danh sách trang `docs`, rồi trả về một list các dict thống nhất.
- **Các trường quan trọng trong mỗi dict**:
  - `chunk_id`: định danh duy nhất gần như theo `product_id`, số trang, tên method (rút gọn), và chỉ số chunk.
  - `page`, `title`: truy vết nguồn khi giảng viên hoặc người học kiểm tra trích dẫn.
  - `chunk_method`: ghi lại đã dùng phương pháp nào — tiện so sánh thí nghiệm.
  - `text`: nội dung đưa vào embedding / retrieval.

---

#### Tại sao schema (metadata) quan trọng?

Trong RAG, sau bước truy hồi, người học cần biết **đoạn này đến từ trang mấy, phương pháp chunk nào**. Giữ `chunk_id`, `page`, `chunk_method`, `text` giúp **truy vết**, **trích dẫn** và **debug** (ví dụ: đổi `sentence_overlap` rồi so sánh lại cùng một `chunk_id` logic) mà không phải sửa toàn bộ pipeline phía sau.

In [9]:
def sentence_split(text: str) -> List[str]:
    text = re.sub(r'\s+', ' ', text.strip())
    parts = re.split(r'(?<=[.!?])\s+', text)
    return [p.strip() for p in parts if p.strip()]


def chunk_sentence_window(text: str, max_words: int = 120, overlap_sentences: int = 1) -> List[str]:
    sents = sentence_split(text)
    chunks, current = [], []
    current_len = 0

    for sent in sents:
        w = len(sent.split())
        if current_len + w <= max_words:
            current.append(sent)
            current_len += w
        else:
            if current:
                chunks.append(' '.join(current))
            current = current[-overlap_sentences:] + [sent] if current else [sent]
            current_len = sum(len(x.split()) for x in current)

    if current:
        chunks.append(' '.join(current))
    return chunks


def chunk_fixed_words(text: str, chunk_words: int = 120, overlap_words: int = 20) -> List[str]:
    words = text.split()
    if not words:
        return []

    step = max(1, chunk_words - overlap_words)
    chunks = []
    for i in range(0, len(words), step):
        chunk = words[i:i + chunk_words]
        if not chunk:
            continue
        chunks.append(' '.join(chunk))
        if i + chunk_words >= len(words):
            break
    return chunks


def chunk_heading_aware(text: str, max_words: int = 140) -> List[str]:
    lines = re.split(r'(?<=[.!?])\s+|\n+', text)
    sections = []
    current = []

    heading_pattern = re.compile(r'^(chapter|section|part|\d+(\.\d+)+)\b', re.IGNORECASE)
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if heading_pattern.match(line) and current:
            sections.append(' '.join(current))
            current = [line]
        else:
            current.append(line)
    if current:
        sections.append(' '.join(current))

    chunks = []
    for sec in sections:
        chunks.extend(chunk_fixed_words(sec, chunk_words=max_words, overlap_words=25))
    return chunks


def build_manual_chunks(docs: List[Dict], method: str, params: Dict) -> List[Dict]:
    out = []
    for doc in docs:
        if method == 'sentence_window':
            chs = chunk_sentence_window(doc['text'], max_words=params['sentence_max_words'], overlap_sentences=params['sentence_overlap'])
        elif method == 'fixed_words':
            chs = chunk_fixed_words(doc['text'], chunk_words=params['fixed_chunk_words'], overlap_words=params['fixed_overlap_words'])
        elif method == 'heading_aware':
            chs = chunk_heading_aware(doc['text'], max_words=params['heading_max_words'])
        else:
            raise ValueError(f'Unknown method: {method}')

        for i, c in enumerate(chs):
            out.append({
                'chunk_id': f"{doc['product_id']}-P{doc['page']}-{method[:3].upper()}-{i}",
                'product_id': doc['product_id'],
                'source': 'manual',
                'page': doc.get('page'),
                'chunk_method': method,
                'text': c,
                'title': doc['title'],
            })
    return out


def print_step2_chunk_preview(
    method_title: str,
    chunks: List[str],
    params_line: str,
    preview_max: int = 4,
    head_chars: int = 320,
) -> None:
    """Readable stdout for Step 2.3-2.5 to compare chunking methods in the log."""
    sep = "=" * 76
    wcounts = [len(c.split()) for c in chunks]
    print(sep)
    print(f"  {method_title}")
    print(f"  {params_line}")
    mn = min(wcounts) if wcounts else 0
    mx = max(wcounts) if wcounts else 0
    avg = float(np.mean(wcounts)) if wcounts else 0.0
    print(
        f"  -> n_chunks={len(chunks)}  |  words/chunk: min={mn}  avg={avg:.1f}  max={mx}"
    )
    print(sep)
    n_show = min(preview_max, len(chunks))
    for i, c in enumerate(chunks[:n_show], start=1):
        wc = len(c.split())
        flat = re.sub(r'\\s+', ' ', c).strip()
        if len(flat) <= head_chars + 80:
            body = flat
        else:
            tail = flat[-140:] if len(flat) > 140 else flat
            body = flat[:head_chars] + "  ...  " + tail
        print(f"\n  --- Preview chunk {i}/{len(chunks)} ({wc} words) ---")
        print(f"  {body}")
    if len(chunks) > n_show:
        more = len(chunks) - n_show
        print(
            f"\n  ... ({more} more chunks not shown; increase preview_max in the call.)"
        )
    print()

### Step 2.2 - Set experiment parameters (interactive tuning)

This parameter cell is your control panel for chunking experiments.

### Parameter meaning
- `sentence_max_words`: max size of sentence-window chunks.
- `sentence_overlap`: number of trailing sentences carried into the next chunk.
- `fixed_chunk_words`: target size for fixed-word chunks.
- `fixed_overlap_words`: overlap size for fixed-word chunks.
- `heading_max_words`: max size after heading-based splitting.

### How to experiment
1. Change one parameter at a time.
2. Rerun Step 2.3 -> Step 2.7.
3. Observe how chunk count and previews change.
4. Keep notes on what improves readability + retrieval alignment.

In [10]:
chunk_params = {
    'sentence_max_words': 120,
    'sentence_overlap': 1,
    'fixed_chunk_words': 120,
    'fixed_overlap_words': 20,
    'heading_max_words': 140,
}

# Pick a page with enough words so Methods A/B/C can differ in the previews (very short pages -> 1 chunk each).
_MIN_WORDS_FOR_SAMPLE = 120
if not manual_docs:
    raise ValueError('manual_docs is empty — run the PDF load cell before Step 2.2.')

_candidates = [d for d in manual_docs if len(d['text'].split()) >= _MIN_WORDS_FOR_SAMPLE]
sample_page = max(
    _candidates or manual_docs,
    key=lambda d: len(d['text'].split()),
)
sample_text = sample_page['text']
_wc = len(sample_text.split())

print('=' * 76)
print('  STEP 2.2 — chunk experiment parameters + sample page for A/B/C previews')
print('=' * 76)
print('\nchunk_params:', chunk_params)
print('\nSample page (for Step 2.3–2.5):')
print('  title :', sample_page['title'])
print('  words :', _wc)
if _wc < _MIN_WORDS_FOR_SAMPLE:
    print(
        f'  note: no page had >= {_MIN_WORDS_FOR_SAMPLE} words; using longest page — '
        'each method may still return a single chunk.'
    )

_w_lens = sorted(len(d['text'].split()) for d in manual_docs)
mid = _w_lens[len(_w_lens) // 2]
print('\nCorpus (all loaded pages) word-count summary:')
print(f'  pages: {len(manual_docs)}  |  min / median / max words per page: {_w_lens[0]} / {mid} / {_w_lens[-1]}')
print('\nRun Step 2.3, then 2.4, then 2.5 and compare the boxed headers + previews in order.')

  STEP 2.2 — chunk experiment parameters + sample page for A/B/C previews

chunk_params: {'sentence_max_words': 120, 'sentence_overlap': 1, 'fixed_chunk_words': 120, 'fixed_overlap_words': 20, 'heading_max_words': 140}

Sample page (for Step 2.3–2.5):
  title : IGF Code - Page 77
  words : 557

Corpus (all loaded pages) word-count summary:
  pages: 124  |  min / median / max words per page: 32 / 375 / 557

Run Step 2.3, then 2.4, then 2.5 and compare the boxed headers + previews in order.


### Step 2.3 - Method A: Sentence-window chunking (visual example)

This method respects sentence boundaries and adds sentence overlap.
It is often a good default for regulatory text.

In [11]:
sample_chunks_sentence = chunk_sentence_window(
    sample_text,
    max_words=chunk_params['sentence_max_words'],
    overlap_sentences=chunk_params['sentence_overlap'],
)

print_step2_chunk_preview(
    'METHOD A — sentence_window (sentence boundaries + sentence overlap)',
    sample_chunks_sentence,
    params_line=(
        f"max_words={chunk_params['sentence_max_words']}, "
        f"overlap_sentences={chunk_params['sentence_overlap']}"
    ),
)

  METHOD A — sentence_window (sentence boundaries + sentence overlap)
  max_words=120, overlap_sentences=1
  -> n_chunks=9  |  words/chunk: min=36  avg=91.4  max=136

  --- Preview chunk 1/9 (64 words) ---
  MSC 95/22/Add.1 Annex 1, page 77 I:\MSC\95\MSC 95-22-ADD.1 (E).docx 9.4.3 The automatic master gas fuel valve shall be operable from safe locations on escape routes inside a machinery space containing a gas consumer, the engine con trol room, if applicable; outside the machinery space, and from the navigation bridge. 9  ...  achinery space, and from the navigation bridge. 9.4.4 Each gas consumer shall be provided with "double block and bleed " valves arrangement.

  --- Preview chunk 2/9 (78 words) ---
  9.4.4 Each gas consumer shall be provided with "double block and bleed " valves arrangement. These valves shall be arranged as outlined in .1 or .2 so that when the safe ty system required in 15.2.2 is activated this will cause the shutoff valves that are in series to close automat

### Step 2.4 - Method B: Fixed-size word chunking (visual example)

This method is simple and deterministic.
Try changing `fixed_chunk_words` and `fixed_overlap_words` to observe boundary behavior.

In [12]:
sample_chunks_fixed = chunk_fixed_words(
    sample_text,
    chunk_words=chunk_params['fixed_chunk_words'],
    overlap_words=chunk_params['fixed_overlap_words'],
)

print_step2_chunk_preview(
    'METHOD B — fixed_words (sliding word windows; may cut mid-sentence)',
    sample_chunks_fixed,
    params_line=(
        f"chunk_words={chunk_params['fixed_chunk_words']}, "
        f"overlap_words={chunk_params['fixed_overlap_words']}"
    ),
)

  METHOD B — fixed_words (sliding word windows; may cut mid-sentence)
  chunk_words=120, overlap_words=20
  -> n_chunks=6  |  words/chunk: min=57  avg=109.5  max=120

  --- Preview chunk 1/6 (120 words) ---
  MSC 95/22/Add.1 Annex 1, page 77 I:\MSC\95\MSC 95-22-ADD.1 (E).docx 9.4.3 The automatic master gas fuel valve shall be operable from safe locations on escape routes inside a machinery space containing a gas consumer, the engine con trol room, if applicable; outside the machinery space, and from the navigation bridge. 9  ...   are in series to close automatically and the bleed valve to open automatically and: .1 the two shutoff valves shall be in series in the gas

  --- Preview chunk 2/6 (120 words) ---
  and the bleed valve to open automatically and: .1 the two shutoff valves shall be in series in the gas fuel pipe to the gas consuming equipment. The bleed valve shall be in a pipe that vents to a safe location in the open air that portion of the gas fuel piping that is between th

In [13]:
# Parameter sensitivity demo (Fixed-size method; same sample_text as Step 2.2)
print('=' * 76)
print('  Step 2.4 add-on — fixed_words sensitivity (two settings on the SAME page)')
print('=' * 76)

fixed_small = chunk_fixed_words(sample_text, chunk_words=80, overlap_words=10)
fixed_large = chunk_fixed_words(sample_text, chunk_words=160, overlap_words=30)

demo_df = pd.DataFrame([
    {'setting': '80 words / 10 overlap', 'n_chunks': len(fixed_small), 'avg_words': np.mean([len(x.split()) for x in fixed_small]) if fixed_small else 0},
    {'setting': '160 words / 30 overlap', 'n_chunks': len(fixed_large), 'avg_words': np.mean([len(x.split()) for x in fixed_large]) if fixed_large else 0},
])

display(demo_df)
print(demo_df.to_string(index=False))
print(
    '\nTip: lower chunk_words -> more chunks; higher overlap -> more text repeated between neighbors.'
)

print_step2_chunk_preview(
    'Sensitivity — fixed_words SMALL (80/10)',
    fixed_small,
    params_line='chunk_words=80, overlap_words=10',
    preview_max=2,
)
print_step2_chunk_preview(
    'Sensitivity — fixed_words LARGE (160/30)',
    fixed_large,
    params_line='chunk_words=160, overlap_words=30',
    preview_max=2,
)

  Step 2.4 add-on — fixed_words sensitivity (two settings on the SAME page)


,setting,n_chunks,avg_words
0,80 words / 10 overlap,8,78.375
1,160 words / 30 overlap,5,135.400


               setting  n_chunks  avg_words
 80 words / 10 overlap         8     78.375
160 words / 30 overlap         5    135.400

Tip: lower chunk_words -> more chunks; higher overlap -> more text repeated between neighbors.
  Sensitivity — fixed_words SMALL (80/10)
  chunk_words=80, overlap_words=10
  -> n_chunks=8  |  words/chunk: min=67  avg=78.4  max=80

  --- Preview chunk 1/8 (80 words) ---
  MSC 95/22/Add.1 Annex 1, page 77 I:\MSC\95\MSC 95-22-ADD.1 (E).docx 9.4.3 The automatic master gas fuel valve shall be operable from safe locations on escape routes inside a machinery space containing a gas consumer, the engine con trol room, if applicable; outside the machinery space, and from the navigation bridge. 9  ...   be provided with "double block and bleed " valves arrangement. These valves shall be arranged as outlined in .1 or .2 so that when the safe

  --- Preview chunk 2/8 (80 words) ---
  outlined in .1 or .2 so that when the safe ty system required in 15.2.2 is activated 

### Step 2.5 - Method C: Heading-aware chunking (visual example)

This method tries to respect document structure by detecting heading-like patterns.

### Why it can help
- Regulatory/standard documents often have chapter/section logic.
- Keeping section-local text together can improve retrieval coherence.

### What to inspect
- Are chunks aligned with section transitions?
- Is important context split too aggressively?
- Compare chunk count against Method A/B.

In [14]:
sample_chunks_heading = chunk_heading_aware(
    sample_text,
    max_words=chunk_params['heading_max_words'],
)

print_step2_chunk_preview(
    'METHOD C — heading_aware (split on heading-like lines, then fixed windows per section)',
    sample_chunks_heading,
    params_line=f"max_words={chunk_params['heading_max_words']} (inner chunk_words)",
)

  METHOD C — heading_aware (split on heading-like lines, then fixed windows per section)
  max_words=140 (inner chunk_words)
  -> n_chunks=11  |  words/chunk: min=17  avg=52.9  max=140

  --- Preview chunk 1/11 (49 words) ---
  MSC 95/22/Add.1 Annex 1, page 77 I:\MSC\95\MSC 95-22-ADD.1 (E).docx 9.4.3 The automatic master gas fuel valve shall be operable from safe locations on escape routes inside a machinery space containing a gas consumer, the engine con trol room, if applicable; outside the machinery space, and from the navigation bridge.

  --- Preview chunk 2/11 (140 words) ---
  9.4.4 Each gas consumer shall be provided with "double block and bleed " valves arrangement. These valves shall be arranged as outlined in .1 or .2 so that when the safe ty system required in 15.2.2 is activated this will cause the shutoff valves that are in series to close automatically and the bleed valve to open aut  ...  he function of one of the shutoff valves in series and the bleed valve can be inco

In [15]:
# Method C sensitivity demo (same sample_text as Step 2.2)
print('=' * 76)
print('  Step 2.5 add-on — heading_aware sensitivity (two max_words on the SAME page)')
print('=' * 76)

heading_small = chunk_heading_aware(sample_text, max_words=100)
heading_large = chunk_heading_aware(sample_text, max_words=180)

heading_demo_df = pd.DataFrame([
    {'setting': 'heading max_words=100', 'n_chunks': len(heading_small), 'avg_words': np.mean([len(x.split()) for x in heading_small]) if heading_small else 0},
    {'setting': 'heading max_words=180', 'n_chunks': len(heading_large), 'avg_words': np.mean([len(x.split()) for x in heading_large]) if heading_large else 0},
])

display(heading_demo_df)
print(heading_demo_df.to_string(index=False))
print('\nTip: smaller max_words usually increases chunk count after sections are formed.')

print_step2_chunk_preview(
    'Sensitivity — heading_aware SMALL (max_words=100)',
    heading_small,
    params_line='max_words=100',
    preview_max=2,
)
print_step2_chunk_preview(
    'Sensitivity — heading_aware LARGE (max_words=180)',
    heading_large,
    params_line='max_words=180',
    preview_max=2,
)

  Step 2.5 add-on — heading_aware sensitivity (two max_words on the SAME page)


,setting,n_chunks,avg_words
0,heading max_words=100,11,52.909091
1,heading max_words=180,10,55.700000


              setting  n_chunks  avg_words
heading max_words=100        11  52.909091
heading max_words=180        10  55.700000

Tip: smaller max_words usually increases chunk count after sections are formed.
  Sensitivity — heading_aware SMALL (max_words=100)
  max_words=100
  -> n_chunks=11  |  words/chunk: min=17  avg=52.9  max=100

  --- Preview chunk 1/11 (49 words) ---
  MSC 95/22/Add.1 Annex 1, page 77 I:\MSC\95\MSC 95-22-ADD.1 (E).docx 9.4.3 The automatic master gas fuel valve shall be operable from safe locations on escape routes inside a machinery space containing a gas consumer, the engine con trol room, if applicable; outside the machinery space, and from the navigation bridge.

  --- Preview chunk 2/11 (100 words) ---
  9.4.4 Each gas consumer shall be provided with "double block and bleed " valves arrangement. These valves shall be arranged as outlined in .1 or .2 so that when the safe ty system required in 15.2.2 is activated this will cause the shutoff valves that are 

### Step 2.6 - Compare all methods (table + quick bar view)

The next code cell compares methods on the **entire loaded corpus** using your current `chunk_params`.

### What the output shows
- A **table** with `n_chunks`, `min_words`, `avg_words`, `median_words`, `max_words` per method.
- The same table printed as **plain text** (easy to copy into lab notes).
- A simple **ASCII bar** for `n_chunks` so differences pop visually.

Use this together with Step 2.3–2.5 previews: **similar chunk counts can still mean different boundaries**.

In [16]:
# (Step 2.6) Full-corpus comparison — same chunk_params for every method
manual_chunks_sentence = build_manual_chunks(manual_docs, method='sentence_window', params=chunk_params)
manual_chunks_fixed = build_manual_chunks(manual_docs, method='fixed_words', params=chunk_params)
manual_chunks_heading = build_manual_chunks(manual_docs, method='heading_aware', params=chunk_params)


def chunk_stats(chunks: List[Dict], method: str) -> Dict:
    lengths = [len(x['text'].split()) for x in chunks]
    return {
        'method': method,
        'n_chunks': len(chunks),
        'min_words': int(np.min(lengths)) if lengths else 0,
        'avg_words': float(np.mean(lengths)) if lengths else 0.0,
        'median_words': float(np.median(lengths)) if lengths else 0.0,
        'max_words': int(np.max(lengths)) if lengths else 0,
    }


_rows = [
    chunk_stats(manual_chunks_sentence, 'sentence_window'),
    chunk_stats(manual_chunks_fixed, 'fixed_words'),
    chunk_stats(manual_chunks_heading, 'heading_aware'),
]
_order = ['sentence_window', 'fixed_words', 'heading_aware']
compare_df = pd.DataFrame(_rows).set_index('method').loc[_order].reset_index()

print('=' * 76)
print('  STEP 2.6 — full corpus: three methods, identical chunk_params')
print('=' * 76)
print('\nTable (all pages in manual_docs):')
display(compare_df)
print(compare_df.round({'avg_words': 1, 'median_words': 1}).to_string(index=False))

_mx = int(compare_df['n_chunks'].max()) or 1
_bar_w = 40
print('\nn_chunks visual (longer bar => more chunks for this corpus + params):')
for _, row in compare_df.iterrows():
    n = int(round(_bar_w * row['n_chunks'] / _mx))
    label = f"{row['method']:<18}"
    print(f"  {label} | {'#' * n}  ({int(row['n_chunks'])})")

print(
    '\nNote: equal n_chunks does not mean equal chunk boundaries — compare Step 2.3–2.5 previews.'
)

  STEP 2.6 — full corpus: three methods, identical chunk_params

Table (all pages in manual_docs):


,method,n_chunks,min_words,avg_words,median_words,max_words
0,sentence_window,604,8,103.486755,106.0,343
1,fixed_words,493,22,106.987830,120.0,120
2,heading_aware,807,1,59.126394,45.0,140


         method  n_chunks  min_words  avg_words  median_words  max_words
sentence_window       604          8      103.5         106.0        343
    fixed_words       493         22      107.0         120.0        120
  heading_aware       807          1       59.1          45.0        140

n_chunks visual (longer bar => more chunks for this corpus + params):
  sentence_window    | ##############################  (604)
  fixed_words        | ########################  (493)
  heading_aware      | ########################################  (807)

Note: equal n_chunks does not mean equal chunk boundaries — compare Step 2.3–2.5 previews.


### Step 2.7 - Select default chunk method for downstream pipeline

This cell determines which chunk set is used in Step 3 onward.

### Recommended workflow
1. Review previews from Methods A/B/C.
2. Review Step 2.6 metrics.
3. Set `DEFAULT_CHUNK_METHOD`.
4. Rerun this cell, then continue to embedding.

You can revisit Step 2 anytime and change this choice.

In [17]:
DEFAULT_CHUNK_METHOD = 'sentence_window'  # try: 'fixed_words' or 'heading_aware'

method_to_chunks = {
    'sentence_window': manual_chunks_sentence,
    'fixed_words': manual_chunks_fixed,
    'heading_aware': manual_chunks_heading,
}
manual_chunks = method_to_chunks[DEFAULT_CHUNK_METHOD]

print(f'Using default chunk method: {DEFAULT_CHUNK_METHOD}')
pd.DataFrame(manual_chunks)[['chunk_id', 'page', 'chunk_method', 'text']].head(8)

Using default chunk method: sentence_window


,chunk_id,page,chunk_method,text
0,IGF-CODE-P1-SEN-0,1,sentence_window,"MSC 95/22/Add.1 Annex 1, page 1 I:\MSC\95\MSC ..."
1,IGF-CODE-P2-SEN-0,2,sentence_window,"MSC 95/22/Add.1 Annex 1, page 2 I:\MSC\95\MSC ..."
2,IGF-CODE-P2-SEN-1,2,sentence_window,14 5.2 Functional requirements ..................
3,IGF-CODE-P3-SEN-0,3,sentence_window,"MSC 95/22/Add.1 Annex 1, page 3 I:\MSC\95\MSC ..."
4,IGF-CODE-P3-SEN-1,3,sentence_window,63 6.11 Regulations on atmosphere control with...
5,IGF-CODE-P3-SEN-2,3,sentence_window,75 8.5 Regulations for bunkering system .........
6,IGF-CODE-P4-SEN-0,4,sentence_window,"MSC 95/22/Add.1 Annex 1, page 4 I:\MSC\95\MSC ..."
7,IGF-CODE-P4-SEN-1,4,sentence_window,84 11.5 Regulations for water spray system ......


## Step 3 - Convert selected chunks to embeddings (deep dive)

Now we move from text preprocessing to vectorization.

### Learning goals
1. Understand how chunk text becomes numeric vectors.
2. See how encoding settings affect speed and quality.
3. Validate embeddings before indexing.

### What this step covers
- Base embedding run from the selected chunk method.
- Optional performance tuning (`batch_size`).
- Sanity diagnostics (shape, norms, NaN checks).
- Semantic preview via query-to-chunk similarity.

### Why this step is critical
If embedding quality is poor, retrieval will be poor even when prompt engineering is strong.
Treat this step as a quality gate before FAISS/Chroma indexing.

In [18]:
# Step 3 code: embed selected chunk set
all_chunks = manual_chunks
print('Selected chunk method:', DEFAULT_CHUNK_METHOD)
print('Total chunks:', len(all_chunks))

embed_model_name = 'sentence-transformers/all-MiniLM-L6-v2'
embedder = SentenceTransformer(embed_model_name)
texts = [c['text'] for c in all_chunks]
embeddings = embedder.encode(texts, normalize_embeddings=True)

print('Embedding model:', embed_model_name)
print('Embedding shape:', embeddings.shape)
print('Vector preview (first 8 dims):', embeddings[0][:8] if len(embeddings) else 'n/a')

Selected chunk method: sentence_window
Total chunks: 604


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6848.01it/s]


Embedding model: sentence-transformers/all-MiniLM-L6-v2
Embedding shape: (604, 384)
Vector preview (first 8 dims): [-0.07050282 -0.00329985 -0.04261405  0.02423807  0.09385647  0.02078965
  0.01468495  0.09177501]


### Step 3.1 - Embedding configuration and performance tuning

The embedding stage has direct impact on both quality and runtime cost.

### Practical considerations
- **Model choice**: larger models may improve semantic quality but increase latency/memory.
- **Batch size**: larger batches are faster on GPU but can cause OOM.
- **Normalization**: with `normalize_embeddings=True`, cosine similarity can be computed using inner product.

### What we do next
We re-run encoding with explicit batch settings so learners can observe runtime and output shape.

In [19]:
import time

BATCH_SIZE = 32  # try 16, 32, 64 depending on your runtime
start_t = time.perf_counter()
embeddings = embedder.encode(
    texts,
    batch_size=BATCH_SIZE,
    normalize_embeddings=True,
    show_progress_bar=False,
)
elapsed = time.perf_counter() - start_t

print(f'Batch size: {BATCH_SIZE}')
print(f'Encoded {len(texts)} chunks in {elapsed:.2f}s')
print('Embedding shape:', embeddings.shape)

Batch size: 32
Encoded 604 chunks in 0.94s
Embedding shape: (604, 384)


## Step 4 - Set up vector search with FAISS

This step builds the primary retrieval engine used by the RAG pipeline.

### What this code does
1. Builds an in-memory FAISS index from embedding vectors.
2. Maps FAISS vector ids back to chunk metadata.
3. Defines `search_faiss(...)` for top-k semantic retrieval.
4. Runs a sample query to verify retrieval quality.

### Important points
- We use `IndexFlatIP` with normalized embeddings (cosine-like behavior).
- Retrieval quality here strongly influences final answer quality in Step 7/9.
- Inspect top results now; fix chunking if results look irrelevant.

In [20]:
norms = np.linalg.norm(embeddings, axis=1)

diag_df = pd.DataFrame({
    'metric': ['num_chunks', 'num_vectors', 'vector_dim', 'norm_min', 'norm_mean', 'norm_max', 'nan_count'],
    'value': [
        len(all_chunks),
        int(embeddings.shape[0]),
        int(embeddings.shape[1]),
        float(norms.min()) if len(norms) else np.nan,
        float(norms.mean()) if len(norms) else np.nan,
        float(norms.max()) if len(norms) else np.nan,
        int(np.isnan(embeddings).sum()),
    ]
})

display(diag_df)
assert embeddings.shape[0] == len(all_chunks), 'Mismatch: vectors vs chunks'
assert int(np.isnan(embeddings).sum()) == 0, 'NaN detected in embeddings'

,metric,value
0,num_chunks,604.0
1,num_vectors,604.0
2,vector_dim,384.0
3,norm_min,1.0
4,norm_mean,1.0
5,norm_max,1.0
6,nan_count,0.0


### Optional Preview - Chroma setup and collection lifecycle

This is a preparatory preview for Step 5.
It separates setup from querying and clarifies collection lifecycle details.

Key ideas:
- create (or reuse) collection safely
- keep metadata schema consistent with FAISS pipeline
- ensure embedding vector shape is valid before insert

In [21]:
if CHROMA_AVAILABLE:
    client = chromadb.Client(Settings(anonymized_telemetry=False))
    collection = client.get_or_create_collection('rag_workshop_collection')

    print('Chroma collection ready:', collection.name)
    print('Current item count (before add):', collection.count())
else:
    print('Chroma not available in this runtime.')

Chroma collection ready: rag_workshop_collection
Current item count (before add): 0


### Optional Preview - Chroma insert and query

This preview cell inserts vectors and executes one test query.

Tip: if you rerun many times, IDs may already exist; in production, use upsert/update strategy.

In [22]:
if CHROMA_AVAILABLE:
    try:
        collection.add(
            ids=[c['chunk_id'] for c in all_chunks],
            documents=[c['text'] for c in all_chunks],
            metadatas=[{'product_id': c['product_id'], 'source': c['source'], 'title': c['title'], 'page': c.get('page')} for c in all_chunks],
            embeddings=[e.tolist() for e in embeddings],
        )
    except Exception as e:
        print('Add warning (possibly duplicate IDs):', e)

    q = 'ventilation and gas detection requirements'
    chroma_res = collection.query(
        query_embeddings=embedder.encode([q], normalize_embeddings=True).tolist(),
        n_results=5,
    )

    chroma_rows = []
    for i in range(len(chroma_res['ids'][0])):
        chroma_rows.append({
            'rank': i + 1,
            'id': chroma_res['ids'][0][i],
            'distance': chroma_res['distances'][0][i],
            'meta': chroma_res['metadatas'][0][i],
        })
    display(pd.DataFrame(chroma_rows))
else:
    print('Skip: Chroma not available.')

,rank,id,distance,meta
0,1,IGF-CODE-P95-SEN-3,0.611286,"{'source': 'manual', 'product_id': 'IGF-CODE',..."
1,2,IGF-CODE-P95-SEN-2,0.628932,"{'page': 95, 'source': 'manual', 'product_id':..."
2,3,IGF-CODE-P95-SEN-1,0.680162,"{'title': 'IGF Code - Page 95', 'page': 95, 's..."
3,4,IGF-CODE-P96-SEN-1,0.682779,"{'source': 'manual', 'product_id': 'IGF-CODE',..."
4,5,IGF-CODE-P78-SEN-2,0.696263,"{'title': 'IGF Code - Page 78', 'product_id': ..."


### Step 4.1 - Build FAISS index

This cell initializes an in-memory FAISS index and loads all chunk vectors.

Key idea:
- We use `IndexFlatIP` with normalized embeddings, which behaves like cosine similarity ranking.

Output checks:
- `index.ntotal` should equal number of chunks.
- metadata map size should match index size.

In [23]:
# Step 4.1 - Build FAISS index and metadata map
dim = embeddings.shape[1]
index = faiss.IndexFlatIP(dim)  # inner product on normalized vectors
index.add(np.array(embeddings).astype('float32'))

id_to_chunk = {i: all_chunks[i] for i in range(len(all_chunks))}

print('Index vectors:', index.ntotal)
print('Vector dimension:', dim)
print('Metadata rows:', len(id_to_chunk))

Index vectors: 604
Vector dimension: 384
Metadata rows: 604


### Optional Preview - Retrieval with page-range filter

The helper `retrieve_context_filtered(...)` over-fetches with `search_faiss`, then keeps hits whose `page` is inside `[min_page, max_page]`. It is **defined in the Step 4.2 code cell** (right after `search_faiss`) so it never runs before the index search API exists.

This pattern is useful later in Step 6. For long PDFs, filtering by page range can reduce noise.

Example use case:
- focus only on operational chapters
- exclude appendices or reference pages

In [24]:
# `retrieve_context_filtered` uses `search_faiss` and is defined in Step 4.2 (next section).
# Run the Step 4.2 code cell — it defines `search_faiss`, then `retrieve_context_filtered`, then the page-range demo.
pass

### Step 4.2 - Define retrieval helper functions

`search_faiss(...)` is the reusable API for semantic retrieval. The same cell also defines `retrieve_context_filtered(...)` (page-range filter) and runs a small demo — it must stay **after** the FAISS index exists from Step 4.1.

Why wrap this in a function?
- Step 6 and Step 9 can call the same retrieval logic.
- Easy to adjust search behavior in one place.
- Keeps notebook flow cleaner and easier to maintain.

In [25]:
# Step 4.2 - Define reusable search utilities

def search_faiss(query: str, top_k: int = 4) -> List[Dict]:
    q_vec = embedder.encode([query], normalize_embeddings=True).astype('float32')
    scores, ids = index.search(q_vec, top_k)

    out = []
    for score, idx in zip(scores[0], ids[0]):
        row = dict(id_to_chunk[int(idx)])
        row['score'] = float(score)
        out.append(row)
    return out


def show_search_results(query: str, top_k: int = 5) -> pd.DataFrame:
    rows = search_faiss(query, top_k=top_k)
    return pd.DataFrame(rows)[['chunk_id', 'page', 'score', 'text']]


def retrieve_context_filtered(
    query: str, top_k: int = 5, min_page: int = None, max_page: int = None
) -> List[Dict]:
    """Semantic search then filter by PDF page range (needs `search_faiss`)."""
    rows = search_faiss(query, top_k=max(15, top_k))
    if min_page is not None:
        rows = [r for r in rows if r.get('page', 0) >= min_page]
    if max_page is not None:
        rows = [r for r in rows if r.get('page', 10**9) <= max_page]
    return rows[:top_k]


# Optional preview: page-range filter (same idea as the old standalone cell above)
filtered_demo = retrieve_context_filtered(
    'ventilation and gas detection requirements',
    top_k=5,
    min_page=1,
    max_page=60,
)
display(pd.DataFrame(filtered_demo)[['chunk_id', 'page', 'score', 'text']].head(5))

,chunk_id,page,score,text
0,IGF-CODE-P12-SEN-1,12,0.594195,"3.2.11 Machinery, systems and components shall..."


### Step 4.3 - Retrieval experiments (query and `top_k` sensitivity)

Use this experiment cell to understand behavior changes as you vary query phrasing and `top_k`.

What to look at:
- Whether top chunks come from relevant pages.
- How fast scores drop after rank 1.
- Whether increasing `top_k` adds useful context or mostly noise.

In [26]:
query = 'What does IGF Code say about ventilation and gas detection requirements?'
print('Baseline top-4:')
display(show_search_results(query, top_k=4).head(4))

experiment_queries = [
    'ventilation and gas detection requirements',
    'operator safety checks for gas systems',
]

rows = []
for q in experiment_queries:
    for k in [3, 5, 8]:
        df = show_search_results(q, top_k=k)
        rows.append({
            'query': q,
            'top_k': k,
            'top_score': float(df['score'].max()) if len(df) else np.nan,
            'kth_score': float(df['score'].iloc[-1]) if len(df) else np.nan,
            'score_drop': float(df['score'].max() - df['score'].iloc[-1]) if len(df) else np.nan,
        })

display(pd.DataFrame(rows))
print('Tip: larger score_drop usually indicates stronger separation between top and tail results.')

Baseline top-4:


,chunk_id,page,score,text
0,IGF-CODE-P1-SEN-0,1,0.580635,"MSC 95/22/Add.1 Annex 1, page 1 I:\MSC\95\MSC ..."
1,IGF-CODE-P87-SEN-3,87,0.572780,12.5.3 Hazardous area zone 225 12.5.3.1 This z...
2,IGF-CODE-P95-SEN-2,95,0.550500,15.8 Regulations for gas detection 15.8.1 Perm...
3,IGF-CODE-P96-SEN-0,96,0.550331,"MSC 95/22/Add.1 Annex 1, page 96 I:\MSC\95\MSC..."


,query,top_k,top_score,kth_score,score_drop
0,ventilation and gas detection requirements,3,0.694357,0.659919,0.034438
1,ventilation and gas detection requirements,5,0.694357,0.651868,0.042488
2,ventilation and gas detection requirements,8,0.694357,0.620520,0.073837
3,operator safety checks for gas systems,3,0.615658,0.601606,0.014053
4,operator safety checks for gas systems,5,0.615658,0.593828,0.021830
5,operator safety checks for gas systems,8,0.615658,0.558686,0.056973


Tip: larger score_drop usually indicates stronger separation between top and tail results.


## Step 5 - Optional vector database: Chroma

FAISS is enough for most local demos. Chroma is useful when you want collection persistence and metadata-oriented workflows.

### This step will
1. Create/get a Chroma collection.
2. Insert documents, metadata, and embeddings.
3. Run one query to verify retrieval.

If Chroma is unavailable, skip this step and continue.

In [27]:
if CHROMA_AVAILABLE:
    client = chromadb.Client(Settings(anonymized_telemetry=False))
    collection = client.get_or_create_collection('rag_workshop_collection')

    collection.add(
        ids=[c['chunk_id'] for c in all_chunks],
        documents=[c['text'] for c in all_chunks],
        metadatas=[{'product_id': c['product_id'], 'source': c['source'], 'title': c['title']} for c in all_chunks],
        embeddings=[e.tolist() for e in embeddings]
    )

    chroma_res = collection.query(
        query_embeddings=embedder.encode([query], normalize_embeddings=True).tolist(),
        n_results=4
    )

    print('Chroma retrieval sample:')
    for i in range(len(chroma_res['ids'][0])):
        print('-', chroma_res['ids'][0][i], '|', chroma_res['metadatas'][0][i], '| score:', chroma_res['distances'][0][i])
else:
    print('Chroma not available. FAISS pipeline is enough for this workshop.')

Chroma retrieval sample:
- IGF-CODE-P1-SEN-0 | {'product_id': 'IGF-CODE', 'title': 'IGF Code - Page 1', 'page': 1, 'source': 'manual'} | score: 0.8387296199798584
- IGF-CODE-P87-SEN-3 | {'page': 87, 'source': 'manual', 'title': 'IGF Code - Page 87', 'product_id': 'IGF-CODE'} | score: 0.8544398546218872
- IGF-CODE-P95-SEN-2 | {'product_id': 'IGF-CODE', 'title': 'IGF Code - Page 95', 'page': 95, 'source': 'manual'} | score: 0.8990009427070618
- IGF-CODE-P96-SEN-0 | {'title': 'IGF Code - Page 96', 'page': 96, 'product_id': 'IGF-CODE', 'source': 'manual'} | score: 0.8993383646011353


## Step 6 - Build a retrieval pipeline over PDF chunks

This step turns raw vector search output into a reusable retrieval interface for prompting.

### Key functions
1. `retrieve_context(...)`: returns top-k chunk candidates.
2. `format_context(...)`: formats retrieved chunks into a prompt-ready context block.

### Why formatting matters
- Structured context makes LLM instructions more consistent.
- Explicit doc numbering (`[Doc 1]`, `[Doc 2]`) enables citation-aware prompting in Step 7.
- Including `page` metadata helps trace responses back to source locations.

In [28]:
def retrieve_context(query: str, top_k: int = 4) -> List[Dict]:
    candidates = search_faiss(query, top_k=max(10, top_k))
    return candidates[:top_k]


def format_context(retrieved: List[Dict]) -> str:
    blocks = []
    for i, r in enumerate(retrieved, start=1):
        page_info = f" page={r.get('page', 'n/a')}" if r.get('source') == 'manual' else ''
        blocks.append(
            f"[Doc {i}] source={r['source']} product={r['product_id']}{page_info} score={r['score']:.3f}\n{r['text']}"
        )
    return '\n\n'.join(blocks)

user_question = 'For IGF compliance, what should operators verify for ventilation and gas detection systems?'
retrieved_docs = retrieve_context(user_question, top_k=5)
context_text = format_context(retrieved_docs)

print(context_text[:3000])

[Doc 1] source=manual product=IGF-CODE page=95 score=0.537
15.8 Regulations for gas detection 15.8.1 Permanently installed gas detectors shall be fitted in: .1 the tank connection spaces; .2 all ducts around fuel pipes; .3 machinery spaces containing gas piping, gas equipment or gas consumers; .4 compressor rooms and fuel preparation rooms; .5 other enclosed spaces containing fuel piping or other fuel equipment without ducting; .6 other enclosed or semi -enclosed spaces where fuel vapours may accumulate including interbarrier spaces and fuel storage hold spaces of independent tanks other than type C; .7 airlocks; .8 gas heating circuit expansion tanks; .9 motor rooms associated with the fuel systems; and .10 or at ventilation inlets to accommodation and machinery spaces if required based on the risk assessment required in 4.2. 15.8.2 In each ESD-protected machinery space, redundant gas detection systems shall be provided.

[Doc 2] source=manual product=IGF-CODE page=95 score=0.536
15.7

## Step 7 - Prompt local LLM (`llama-cpp-python`) with retrieved context

This step converts retrieval output into grounded generation.

### Prompt design goals
- Force answers to stay inside retrieved evidence.
- Require citations (`[Doc i]`) for traceability.
- Return uncertainty when context is insufficient.

### Code focus
- `build_grounded_prompt(...)`: enforces strict answer policy.
- `build_llama_cpp(...)`: initializes local GGUF model with runtime settings.
- `call_local_llm(...)`: executes completion and returns raw model response.

If your model is not loaded, fix `LLAMA_CPP_GGUF` first before proceeding.

In [29]:
# Whether llama-cpp-python is available (used by `build_llama_cpp`).
try:
    from llama_cpp import Llama  # noqa: F401
    LLAMA_CPP_AVAILABLE = True
except ImportError:
    LLAMA_CPP_AVAILABLE = False

def build_grounded_prompt(question: str, context: str) -> str:
    return f"""
You are a maritime technical compliance assistant.
Use ONLY the retrieved context to answer.
If context is insufficient, reply exactly:
I don't have enough evidence from retrieved documents.
Always cite supporting documents as [Doc 1], [Doc 2], ...
Do not invent regulations, clause numbers, or operational steps.

Retrieved Context:
{context}

User Question:
{question}

Return format:
1) Final Answer
2) Evidence Used
3) Confidence (High/Medium/Low)
""".strip()


def build_llama_cpp():
    if not LLAMA_CPP_AVAILABLE:
        return None

    model_path = os.environ.get('LLAMA_CPP_GGUF', '').strip()
    if not model_path:
        return None
    if not Path(model_path).exists():
        return None

    # Colab tip: set env LLAMA_CPP_N_GPU_LAYERS (e.g., 20) when GPU runtime is enabled.
    n_gpu_layers = int(os.environ.get('LLAMA_CPP_N_GPU_LAYERS', '0'))

    return Llama(
        model_path=model_path,
        n_ctx=4096,
        n_threads=max(2, (os.cpu_count() or 4) // 2),
        n_gpu_layers=n_gpu_layers,
        n_batch=512,
        verbose=False,
    )


def call_local_llm(prompt: str, max_tokens: int = 320, temperature: float = 0.2) -> str:
    llm = build_llama_cpp()
    if llm is None:
        return (
            "I don't have enough evidence from retrieved documents. "
            "(Local model is not ready: set LLAMA_CPP_GGUF or run Step 0b.)"
        )

    out = llm.create_completion(
        prompt=prompt,
        max_tokens=max_tokens,
        temperature=temperature,
        stop=['\n\nUser Question:', '\n\nRetrieved Context:']
    )
    return out['choices'][0]['text'].strip()

### Step 7a - Build prompt and run local model

This cell runs your local GGUF model. It uses a **higher `max_tokens`** than the old default so the answer is long enough to read and compare.

**Output:** prompt excerpt, then **stats**, then a **scrollable HTML panel** with the **full** answer (not just the first lines), plus a plain-text head/tail echo for logs.

In [30]:
prompt = build_grounded_prompt(user_question, context_text)
print(prompt[:1200])

sample_llm_answer = call_local_llm(prompt)
print('\n--- Local LLM output ---\n')
print(sample_llm_answer[:2000])

You are a maritime technical compliance assistant.
Use ONLY the retrieved context to answer.
If context is insufficient, reply exactly:
I don't have enough evidence from retrieved documents.
Always cite supporting documents as [Doc 1], [Doc 2], ...
Do not invent regulations, clause numbers, or operational steps.

Retrieved Context:
[Doc 1] source=manual product=IGF-CODE page=95 score=0.537
15.8 Regulations for gas detection 15.8.1 Permanently installed gas detectors shall be fitted in: .1 the tank connection spaces; .2 all ducts around fuel pipes; .3 machinery spaces containing gas piping, gas equipment or gas consumers; .4 compressor rooms and fuel preparation rooms; .5 other enclosed spaces containing fuel piping or other fuel equipment without ducting; .6 other enclosed or semi -enclosed spaces where fuel vapours may accumulate including interbarrier spaces and fuel storage hold spaces of independent tanks other than type C; .7 airlocks; .8 gas heating circuit expansion tanks; .9 mo

llama_context: n_ctx_seq (4096) > n_ctx_train (2048) -- possible training context overflow



--- Local LLM output ---

Example:
Final Answer:
1) Final Answer:
Evidence Used:
1) Confidence (High):


### Optional Utility - Prompt temperature experiment

Run the **next** code cell after Step 7a (and the optional citation cells above if you ran them). `prompt` and `call_local_llm` must already exist.

**Output:** bảng tóm tắt theo từng `temperature` (độ dài + preview 4000 ký tự), gọi model với **`max_tokens = 640`**.

**Lưu kết quả / Persist:** sau khi chạy, `experiment_answers` được ghi vào `data/step7_temperature_experiment.json` (đầy đủ, gồm `answer_full`) và `data/step7_temperature_experiment.csv` (cột preview).

Generate answers with different temperatures to observe stability vs creativity.
In RAG support use cases, lower temperature is usually preferred for consistency.


In [33]:
import json
from datetime import datetime, timezone
from pathlib import Path


temps = [0.0, 0.2, 0.6]
TEMP_MAX_TOKENS = 640

experiment_answers = []

for t in temps:
    ans = call_local_llm(prompt, max_tokens=TEMP_MAX_TOKENS, temperature=t)
    experiment_answers.append(
        {
            'temperature': t,
            'chars': len(ans),
            'words': len(ans.split()),
            'answer_preview': ans[:4000],
            'answer_full': ans,
        }
    )


display(
    pd.DataFrame(
        [
            {k: v for k, v in r.items() if k != 'answer_full'}
            for r in experiment_answers
        ]
    )
)

_out_dir = Path('data')
_out_dir.mkdir(parents=True, exist_ok=True)

_payload = {
    'saved_at_utc': datetime.now(timezone.utc).isoformat(),
    'notebook_section': 'Section A — Optional after Step 7a (temperature)',
    'temps': temps,
    'max_tokens': TEMP_MAX_TOKENS,
    'experiment_answers': experiment_answers,
}

_json_path = _out_dir / 'step7_temperature_experiment.json'
with open(_json_path, 'w', encoding='utf-8') as f:
    json.dump(_payload, f, ensure_ascii=False, indent=2)

_csv_path = _out_dir / 'step7_temperature_experiment.csv'
pd.DataFrame(
    [
        {
            'temperature': r['temperature'],
            'chars': r['chars'],
            'words': r['words'],
            'answer_preview': r['answer_preview'],
        }
        for r in experiment_answers
    ]
).to_csv(_csv_path, index=False, encoding='utf-8')

print('\nSaved experiment_answers / Saved experiment_answers:')
print('  JSON (full text):', _json_path.resolve())
print('  CSV (preview cols):', _csv_path.resolve())


llama_context: n_ctx_seq (4096) > n_ctx_train (2048) -- possible training context overflow
llama_context: n_ctx_seq (4096) > n_ctx_train (2048) -- possible training context overflow
llama_context: n_ctx_seq (4096) > n_ctx_train (2048) -- possible training context overflow


,temperature,chars,words,answer_preview
0,0.0,76,11,Example:\nFinal Answer:\n1) Final Answer:\nEvi...
1,0.2,76,11,Example:\nFinal Answer:\n1) Final Answer:\nEvi...
2,0.6,561,80,Example:\nFinal Answer:\n1) Final Answer:\nEvi...



Saved experiment_answers / Saved experiment_answers:
  JSON (full text): /home/ethan/newgen/KMOU_Course/Module_4/data/step7_temperature_experiment.json
  CSV (preview cols): /home/ethan/newgen/KMOU_Course/Module_4/data/step7_temperature_experiment.csv


## Step 8 - Handling hallucination in responses

Even with RAG, hallucination can still occur when prompts are weak or retrieval misses key evidence.

### Common failure patterns
- Unsupported claims not present in retrieved chunks.
- Overconfident answers from partial evidence.
- Missing citation despite instruction.

### Mitigation flow used here
1. Constrained prompt policy (from Step 7).
2. Required citation format.
3. Lightweight lexical evidence check (`simple_evidence_check`).
4. Safe fallback response when support is weak.

This is a practical baseline. In production, consider stronger faithfulness checks.

In [34]:
def simple_evidence_check(answer: str, retrieved: List[Dict], min_overlap_terms: int = 3) -> Dict:
    """
    Lightweight workshop checker:
    - token overlap between answer and retrieved context
    - warns if overlap is too low
    """
    answer_terms = set(re.findall(r'[a-zA-Z0-9-]+', answer.lower()))
    ctx_terms = set()
    for r in retrieved:
        ctx_terms.update(re.findall(r'[a-zA-Z0-9-]+', r['text'].lower()))

    overlap = answer_terms.intersection(ctx_terms)
    ok = len(overlap) >= min_overlap_terms
    return {
        'supported': ok,
        'overlap_count': len(overlap),
        'sample_overlap_terms': sorted(list(overlap))[:15]
    }

check = simple_evidence_check(sample_llm_answer, retrieved_docs)
print(json.dumps(check, indent=2))

if not check['supported']:
    print("\nFallback: I don't have enough evidence from retrieved documents.")

{
  "supported": false,
  "overlap_count": 1,
  "sample_overlap_terms": [
    "1"
  ]
}

Fallback: I don't have enough evidence from retrieved documents.


## Step 9 - End-to-end RAG function (retrieval + local generation + safety)

This function combines:
1. Retrieval from vector DB
2. Prompt construction
3. Local LLM generation via `llama-cpp-python`
4. Hallucination safety check

You can later swap in another LLM backend if needed.

In [35]:
def rag_answer(question: str, top_k: int = 5) -> Dict:
    retrieved = retrieve_context(question, top_k=top_k)
    context = format_context(retrieved)
    prompt = build_grounded_prompt(question, context)

    generated = call_local_llm(prompt)

    safety = simple_evidence_check(generated, retrieved)
    if not safety['supported']:
        generated = "I don't have enough evidence from retrieved documents. Please provide a more specific IGF query."

    return {
        'question': question,
        'retrieved': retrieved,
        'prompt': prompt,
        'answer': generated,
        'safety': safety,
    }

final_result = rag_answer(
    question='Summarize key operator checks for ventilation and gas detection based on IGF Code context.',
    top_k=5,
)

print('Answer:\n', final_result['answer'])
print('\nSafety check:', final_result['safety'])

llama_context: n_ctx_seq (4096) > n_ctx_train (2048) -- possible training context overflow


Answer:
 Example Answer:
1) Final Answer:
The IGF Code context requires operator checks for ventilation and gas detection. The following key operator checks are used:

1.1.1.1 Regular inspection of ventilation systems and gas detection equipment
1.1.1.2 Regular inspection of gas detection equipment
1.1.1.3 Regular inspection of ventilation systems
1.1.2 Regular inspection of gas detection equipment
1.1.2.1 Regular inspection of gas detection equipment
1.1.2.2 Regular inspection of ventilation systems
1.1.3 Regular inspection of gas detection equipment
1.1.3.1 Regular inspection of gas detection equipment
1.1.3.2 Regular inspection of ventilation systems
1.1.4 Regular inspection of gas detection equipment
1.1.4.1 Regular inspection of gas detection equipment
1.1.4.2 Regular inspection of ventilation systems
1.2.1 Regular inspection of gas detection equipment
1.2.2 Regular inspection of ventilation systems
1.2.3 Regular inspection of gas detection equipment
1.2.4 Regular inspection of ve

### Optional Utility - Citation format check

Run the **next** code cell **immediately after** Step 7a so `sample_llm_answer` already exists (this optional block is placed directly under the Step 7a code cell).

Besides overlap checking, this validates whether the answer includes citations like `[Doc 1]`.
This is a lightweight guardrail for grounded responses.

In [36]:
def citation_format_check(answer: str) -> Dict:
    cites = re.findall(r'\[Doc\s+\d+\]', answer)
    return {
        'has_citation': len(cites) > 0,
        'citation_count': len(cites),
        'citations_found': cites[:10],
    }

cite_check = citation_format_check(sample_llm_answer)
print(json.dumps(cite_check, indent=2))

if not cite_check['has_citation']:
    print('Warning: answer has no explicit [Doc i] citations.')

{
  "has_citation": false,
  "citation_count": 0,
  "citations_found": []
}


## Step 10 - Practice tasks for learners

Use these exercises to deepen understanding and evaluate design trade-offs.

1. Add more maritime PDFs and keep source metadata (`doc_name`, `page`, `section`).
2. Compare retrieval quality for chunk sizes: 80 vs 120 vs 180 words.
3. Add metadata filtering by chapter/page range.
4. Try different local GGUF models and compare answer quality.
5. Measure hallucination rate before/after grounded prompt + evidence check.

---

## Recap

You completed a full RAG workflow with local inference:
- PDF ingestion and chunk design
- Vector indexing and retrieval over PDF chunks
- Grounded local generation with `llama-cpp-python`
- Basic hallucination mitigation and safety fallback

This notebook is now structured as a teaching lab and can be extended into a production-oriented prototype.

---

## Section B - Use Ollama model service with notebook RAG

Important design rule for Section B:
- **RAG logic stays in this notebook** (chunking, embedding, retrieval, context formatting).
- A separate Python file is used **only** to call Ollama models.

Dedicated Ollama client file:
- `ollama_model_runner.py`

This mirrors real-world architecture:
1. Notebook prepares prompt from retrieved context.
2. External client script calls one or multiple Ollama models.
3. Notebook visualizes model responses.

### Step 1: Install lightweight client dependencies

These dependencies are only for calling Ollama API and parsing responses in notebook.
They do not replace notebook RAG steps.

In [37]:
%pip install -q requests sentence-transformers faiss-cpu pypdf

Note: you may need to restart the kernel to use updated packages.


In [38]:
from pathlib import Path

runner_path = Path('ollama_model_runner.py')
print('Runner exists:', runner_path.exists())
print('Runner path:', runner_path.resolve())
if not runner_path.exists():
    raise FileNotFoundError('ollama_model_runner.py not found.')

Runner exists: True
Runner path: /home/ethan/newgen/KMOU_Course/Module_4/ollama_model_runner.py


### Step 2: Install and prepare Ollama service/models (manual checklist)

Install Ollama first (outside notebook):

- **Windows**: Download and install from [https://ollama.com/download](https://ollama.com/download)
- **macOS**: `brew install ollama` or install from Ollama website
- **Linux**:
  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```

Then run these commands in a terminal:

```bash
ollama serve
ollama pull llama3.1:8b
ollama pull mistral:7b
ollama list
```

Why outside notebook:
- Ollama is a persistent service process.
- Notebook kernels may restart/interrupt background services.
- External service + script call is more stable for repeated experiments.

After installation and service startup are complete, continue the cells below.

In [40]:
import sys
import tempfile

OLLAMA_HOST = 'http://localhost:11434'
OLLAMA_MODELS = 'llama3.1:8b'  # edit this mistral:7b
OLLAMA_TEMPERATURE = 0.2
OLLAMA_MAX_TOKENS = 320

# RAG remains in notebook: build prompt from retrieved context created in Step 6
if 'user_question' not in globals() or 'context_text' not in globals():
    raise RuntimeError('Please run Step 6 first to produce user_question and context_text.')

ollama_prompt = build_grounded_prompt(user_question, context_text)

prompt_file = tempfile.NamedTemporaryFile(mode='w', suffix='.txt', delete=False, encoding='utf-8')
prompt_file.write(ollama_prompt)
prompt_file.flush()
prompt_file.close()

cmd = [
    sys.executable,
    'ollama_model_runner.py',
    '--host', OLLAMA_HOST,
    '--models', OLLAMA_MODELS,
    '--prompt-file', prompt_file.name,
    '--temperature', str(OLLAMA_TEMPERATURE),
    '--max-tokens', str(OLLAMA_MAX_TOKENS),
]

print('Prompt chars:', len(ollama_prompt))
print('Command preview:')
print(' '.join(cmd))

Prompt chars: 4936
Command preview:
/home/ethan/anaconda3/envs/kmouc/bin/python ollama_model_runner.py --host http://localhost:11434 --models llama3.1:8b --prompt-file /tmp/tmp2ty0sd07.txt --temperature 0.2 --max-tokens 320


In [41]:
import subprocess

run = subprocess.run(cmd, capture_output=True, text=True)
print('Return code:', run.returncode)
print('\n--- STDOUT (first 2500 chars) ---\n')
print(run.stdout[:2500])

if run.stderr.strip():
    print('\n--- STDERR (first 1200 chars) ---\n')
    print(run.stderr[:1200])

Return code: 0

--- STDOUT (first 2500 chars) ---

{
  "host": "http://localhost:11434",
  "models": [
    "llama3.1:8b"
  ],
  "temperature": 0.2,
  "max_tokens": 320,
  "prompt_chars": 4936,
  "outputs": [
    {
      "model": "llama3.1:8b",
      "response": "**Final Answer**\nOperators should verify that the ventilation and gas detection systems are functioning correctly and in accordance with the regulations.\n\n**Evidence Used**\n[Doc 4] page=92, section 15.3 Regulations – General: \"the safety systems including the field instrumentation shall be arranged to avoid spurious shutdown, e.g. as a result of a faulty gas detector or a wire break in a sensor loop;\" and [Doc 5] page=90, sections 13.6.3 and 13.8.1.\n\n**Confidence**\nHigh",
      "done": true,
      "total_duration": 2528451510,
      "eval_count": 114,
      "status": "ok"
    }
  ]
}



### Step 3: Execute model client with notebook-generated prompt

This runs the external client script.
RAG context/prompt is prepared in notebook, then sent to Ollama through the Python client.

In [42]:
import subprocess
import time

# Safety checks before execution
if 'cmd' not in globals():
    raise RuntimeError('Please run Step 2 code cells first to prepare cmd.')

print('Executing command:')
print(' '.join(cmd))

start_ts = time.time()
run = subprocess.run(cmd, capture_output=True, text=True)
elapsed_sec = time.time() - start_ts

print(f'Return code: {run.returncode}')
print(f'Elapsed: {elapsed_sec:.2f} sec')
print(f'STDOUT chars: {len(run.stdout)} | STDERR chars: {len(run.stderr)}')

Executing command:
/home/ethan/anaconda3/envs/kmouc/bin/python ollama_model_runner.py --host http://localhost:11434 --models llama3.1:8b --prompt-file /tmp/tmp2ty0sd07.txt --temperature 0.2 --max-tokens 320
Return code: 0
Elapsed: 2.90 sec
STDOUT chars: 996 | STDERR chars: 0


In [43]:
# Preview logs for quick debugging when a model call fails
stdout_preview_chars = 3000
stderr_preview_chars = 1500

print('--- STDOUT PREVIEW ---')
print(run.stdout[:stdout_preview_chars] if run.stdout else '(empty stdout)')

print('\n--- STDERR PREVIEW ---')
print(run.stderr[:stderr_preview_chars] if run.stderr else '(empty stderr)')

--- STDOUT PREVIEW ---
{
  "host": "http://localhost:11434",
  "models": [
    "llama3.1:8b"
  ],
  "temperature": 0.2,
  "max_tokens": 320,
  "prompt_chars": 4936,
  "outputs": [
    {
      "model": "llama3.1:8b",
      "response": "**Final Answer**\nOperators should verify that the ventilation and gas detection systems are functioning correctly to ensure a safe management of the whole fuel-gas equipment.\n\n**Evidence Used**\n[Doc 4] source=manual product=IGF-CODE page=92 score=0.515: \"Suitable instrumentation devices shall be fitted to allow a local and a remote reading of essential parameters to ensure a safe management of the whole fuel-gas equipment including bunkering.\"\n\n**Confidence**\nMedium\n\nNote: The evidence provided does not specifically mention ventilation and gas detection systems, but it mentions the importance of suitable instrumentation devices for safe management.",
      "done": true,
      "total_duration": 2755488839,
      "eval_count": 127,
      "status"

### Step 4: Parse JSON summary for visual comparison

The script prints a JSON summary at the end.
This cell extracts and visualizes key fields for easy reading in notebook.

In [44]:
import json

parsed = None
parse_error = None

try:
    parsed = json.loads(run.stdout)
except Exception as e:
    parse_error = e

if parsed is None:
    print('Failed to parse JSON output from ollama_model_runner.py')
    print('Error:', parse_error)
    print('\nRaw stdout preview:')
    print(run.stdout[:1500])
else:
    print('Top-level keys:', list(parsed.keys()))
    print('Host:', parsed.get('host'))
    print('Models:', parsed.get('models'))
    print('Prompt chars:', parsed.get('prompt_chars'))

    out_df = pd.DataFrame(parsed.get('outputs', []))
    if out_df.empty:
        print('No model outputs found in parsed payload.')
    else:
        out_df['response_preview'] = out_df['response'].astype(str).str.slice(0, 280)
        out_df['response_len'] = out_df['response'].astype(str).str.len()

        cols = [c for c in ['model', 'status', 'done', 'eval_count', 'total_duration', 'response_len', 'response_preview'] if c in out_df.columns]
        display(out_df[cols])

Top-level keys: ['host', 'models', 'temperature', 'max_tokens', 'prompt_chars', 'outputs']
Host: http://localhost:11434
Models: ['llama3.1:8b']
Prompt chars: 4936


,model,status,done,eval_count,total_duration,response_len,response_preview
0,llama3.1:8b,ok,True,127,2755488839,656,**Final Answer**\nOperators should verify that...


In [45]:
# Optional quality summary for quick classroom discussion
if 'out_df' in globals() and isinstance(out_df, pd.DataFrame) and not out_df.empty:
    summary_rows = []
    for _, row in out_df.iterrows():
        response_text = str(row.get('response', ''))
        summary_rows.append({
            'model': row.get('model'),
            'status': row.get('status'),
            'response_len': len(response_text),
            'has_citation_like_token': ('[p.' in response_text.lower()) or ('page' in response_text.lower()),
            'mentions_insufficient_context': 'insufficient context' in response_text.lower(),
        })
    display(pd.DataFrame(summary_rows))
else:
    print('Run Step 4 parsing cell first to generate out_df.')

,model,status,response_len,has_citation_like_token,mentions_insufficient_context
0,llama3.1:8b,ok,656,True,False


### Step 5: Re-run with different model sets and compare

Use this template cell to compare multiple runs quickly.
Edit the model sets and rerun.

In [46]:
import json

parsed = None
try:
    parsed = json.loads(run.stdout)
except Exception:
    parsed = None

if parsed is None:
    print('Failed to parse JSON output from ollama_model_runner.py')
    print('Raw stdout preview:')
    print(run.stdout[:1200])
else:
    print('Parsed keys:', list(parsed.keys()))
    print('Models:', parsed.get('models'))
    print('Prompt chars:', parsed.get('prompt_chars'))

    out_df = pd.DataFrame(parsed.get('outputs', []))
    if not out_df.empty:
        out_df['response_preview'] = out_df['response'].astype(str).str.slice(0, 260)
        display(out_df[['model', 'status', 'response_preview', 'eval_count']])

Parsed keys: ['host', 'models', 'temperature', 'max_tokens', 'prompt_chars', 'outputs']
Models: ['llama3.1:8b']
Prompt chars: 4936


,model,status,response_preview,eval_count
0,llama3.1:8b,ok,**Final Answer**\nOperators should verify that...,127


### Troubleshooting checklist

If execution fails:
1. Ensure Ollama server is running: `ollama serve`.
2. Ensure model exists: `ollama list`.
3. Pull missing model: `ollama pull <model>`.
4. Check host and port (`http://localhost:11434`).
5. Re-run Step 3 and inspect stderr.

This separate-runner architecture is intentional and more robust than trying to host Ollama directly inside notebook runtime.

In [47]:
model_sets = [
    'llama3.1:8b',
    'mistral:7b',
    'llama3.1:8b,mistral:7b',
]

compare_rows = []
for model_set in model_sets:
    cmd2 = [
        sys.executable,
        'ollama_model_runner.py',
        '--host', OLLAMA_HOST,
        '--models', model_set,
        '--prompt-file', prompt_file.name,
        '--temperature', str(OLLAMA_TEMPERATURE),
        '--max-tokens', str(OLLAMA_MAX_TOKENS),
    ]
    r = subprocess.run(cmd2, capture_output=True, text=True)
    try:
        payload = json.loads(r.stdout)
        ok_count = sum(1 for o in payload.get('outputs', []) if str(o.get('status', '')).startswith('ok'))
    except Exception:
        ok_count = 0
    compare_rows.append({
        'models': model_set,
        'return_code': r.returncode,
        'ok_outputs': ok_count,
        'stdout_len': len(r.stdout),
        'stderr_len': len(r.stderr),
    })

display(pd.DataFrame(compare_rows))

,models,return_code,ok_outputs,stdout_len,stderr_len
0,llama3.1:8b,0,1,664,0
1,mistral:7b,0,0,323,0
2,"llama3.1:8b,mistral:7b",0,1,1162,0


---

## Section C - Advanced RAG in LangChain

### Why this section exists (for beginners)

In **Section A** you built RAG “by hand”: you chunked text, embedded it, stored vectors in FAISS, wrote retrieval functions, and called the LLM with a prompt. That is the best way to *understand* each moving part.

**LangChain** is a library that wraps common RAG patterns into reusable pieces (documents, embeddings, vector stores, retrievers, chains). You still do the same math (similarity search on vectors), but you spend less boilerplate wiring objects together and you can swap retrieval strategies more quickly.

This section does **not** replace Section A. It **reuses** the same `manual_chunks` you already created so you can compare “raw pipeline” vs “LangChain orchestration” on identical data.

### Prerequisites (run these first)

1. Complete **Section A** through chunking so `manual_chunks` exists and is non-empty.
2. Ideally complete **embedding setup** in Section A as well, because we reuse `embed_model_name` so LangChain uses the **same** Sentence-Transformers model name as the rest of the notebook (fair comparison).
3. For the QA cells that call `call_local_llm(...)`, complete **local LLM setup** in Section A (so generation works the same way as before).

### What you will practice here

- **Documents and metadata**: turning each chunk into a LangChain `Document` with `page_content` and `metadata` (page number, chunk id).
- **Vector store**: building a LangChain FAISS wrapper over those documents (a second index, parallel to your manual FAISS work).
- **Retrievers**: comparing **similarity** search vs **MMR** (diversity-aware retrieval).
- **End-to-end QA**: retrieve with LangChain, then still generate with your notebook’s `call_local_llm` so only the *retrieval layer* changes between experiments.

### Learning goal

By the end of Section C you should be able to explain: *What does LangChain add on top of plain Python + FAISS?* and *When might MMR change the evidence the model sees?*

### Step 1: Install LangChain packages

LangChain is split into multiple installable packages. For this workshop we install:

- **`langchain`**: core abstractions and composition utilities used across examples.
- **`langchain-community`**: community integrations, including the **FAISS vector store wrapper** we use here (`langchain_community.vectorstores.FAISS`).
- **`langchain-huggingface`**: the **`HuggingFaceEmbeddings`** wrapper so LangChain can call the same family of Sentence-Transformers models you already use elsewhere in the notebook.

#### What happens when you run the pip cell?

- Packages download into your current Python environment (local machine, Colab VM, etc.).
- If versions change, you may see a message suggesting a **kernel restart** after installs. If imports fail right after installing, restart the kernel once and rerun imports.

#### Common beginner issues

- **Slow first import**: HuggingFace/Sentence-Transformers may download model weights on first use.
- **Disk / RAM**: embedding the full PDF twice (Section A index + LangChain index) uses extra memory; that is normal for a teaching notebook.

After this step succeeds, continue to Step 2 where we convert `manual_chunks` into LangChain documents.

In [50]:
%pip install -q langchain langchain-community langchain-huggingface

Note: you may need to restart the kernel to use updated packages.


### Step 2: Build LangChain documents and vector store

#### Mental model: from `manual_chunks` to LangChain objects

Your `manual_chunks` list is already a clean “database table” of text snippets. LangChain expects each row to become a **`Document`**:

- **`page_content`**: the actual chunk text the retriever will rank and the LLM will read.
- **`metadata`**: structured fields you can filter on later (here: `chunk_id`, `page`, `chunk_method`). Good metadata makes debugging much easier because you can show *which PDF page* supported an answer.

#### What the code cell does (read this before running)

1. **`Document(...)`**: wraps each dict in `manual_chunks` into LangChain’s standard type.
2. **`HuggingFaceEmbeddings(model_name=...)`**: tells LangChain how to turn strings into vectors. We intentionally reuse `embed_model_name` from Section A when it exists, so embeddings are comparable.
3. **`FAISS.from_documents(...)`** (imported as `LCFAISS` in code): LangChain builds an **in-memory FAISS index** from your documents. This is parallel to the FAISS index you built manually; it does **not delete** your earlier work.

#### Why build a second index?

LangChain retrievers are easiest to demo when the vector store is a LangChain object. **Pedagogically**, you can think of this as: *same chunks, same embedding idea, different “API surface” for retrieval experiments.*

#### Expected output

You should see printed counts like number of LangChain docs and the embedding model name. If you get `RuntimeError` about missing `manual_chunks`, go back to Section A chunking cells.

In [51]:
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS as LCFAISS

if 'manual_chunks' not in globals() or not manual_chunks:
    raise RuntimeError('Please run Section A up to chunking step to create manual_chunks.')

lc_docs = [
    Document(
        page_content=c['text'],
        metadata={
            'chunk_id': c.get('chunk_id'),
            'page': c.get('page'),
            'chunk_method': c.get('chunk_method')
        }
    )
    for c in manual_chunks
]

lc_embedding_model_name = globals().get('embed_model_name', 'sentence-transformers/all-MiniLM-L6-v2')
lc_embeddings = HuggingFaceEmbeddings(model_name=lc_embedding_model_name)

lc_vectorstore = LCFAISS.from_documents(lc_docs, lc_embeddings)
print('LangChain docs:', len(lc_docs))
print('Embedding model:', lc_embedding_model_name)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6023.44it/s]


LangChain docs: 604
Embedding model: sentence-transformers/all-MiniLM-L6-v2


### Step 3: Advanced retriever setup (Similarity vs MMR)

Retrieval is the step that decides **which evidence** the LLM is allowed to see. Small changes here can change answers a lot, even if the model stays the same.

#### Strategy A: Similarity search (baseline)

**Idea**: rank chunks by embedding similarity to the query and return the top `k`.

- **Strength**: simple, fast, usually good when the “right answer” is concentrated in a few highly similar passages.
- **Weakness**: the top results can be **near duplicates** (very similar chunks from adjacent pages). That wastes context window and can bias the model toward repeating the same idea.

In LangChain this is `search_type='similarity'`.

#### Strategy B: MMR (Max Marginal Relevance)

**Idea**: start from a pool of highly similar candidates (`fetch_k`), then pick `k` chunks while balancing:

- relevance to the query, and
- **diversity** among selected chunks (avoid picking five almost-identical paragraphs).

`lambda_mult` controls the trade-off (closer to 1.0 tends to favor relevance; closer to 0.0 favors diversity—interpretation can vary slightly by implementation, so treat it as a tuning knob).

In LangChain this is `search_type='mmr'`.

#### How to read the table output

The code cell runs **the same demo query** through both retrievers and prints a combined preview table. Compare:

- **Pages**: does MMR spread evidence across more pages?
- **Text previews**: does MMR reduce repetitive wording?

#### Parameters you can tweak (learning exercise)

- `k`: how many chunks enter the prompt context.
- `fetch_k` (MMR): how many candidates to consider before diversifying.
- `lambda_mult` (MMR): relevance vs diversity emphasis.

In [52]:
lc_retriever_similarity = lc_vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 4}
)

lc_retriever_mmr = lc_vectorstore.as_retriever(
    search_type='mmr',
    search_kwargs={'k': 4, 'fetch_k': 12, 'lambda_mult': 0.5}
)

lc_demo_query = globals().get('user_question', 'What does IGF Code mention about fire safety arrangements?')

sim_docs = lc_retriever_similarity.invoke(lc_demo_query)
mmr_docs = lc_retriever_mmr.invoke(lc_demo_query)

def _doc_preview(rows, label):
    out = []
    for i, d in enumerate(rows, 1):
        out.append({
            'mode': label,
            'rank': i,
            'page': d.metadata.get('page'),
            'chunk_id': d.metadata.get('chunk_id'),
            'text_preview': d.page_content[:180].replace('\n', ' ')
        })
    return out

display(pd.DataFrame(_doc_preview(sim_docs, 'similarity') + _doc_preview(mmr_docs, 'mmr')))

,mode,rank,page,chunk_id,text_preview
0,similarity,1,95,IGF-CODE-P95-SEN-2,15.8 Regulations for gas detection 15.8.1 Perm...
1,similarity,2,95,IGF-CODE-P95-SEN-1,15.7 Regulations for gas engine monitoring In ...
2,similarity,3,96,IGF-CODE-P96-SEN-1,15.8.8 Audible and visible alarms from the gas...
3,similarity,4,92,IGF-CODE-P92-SEN-4,This includes power supplies and input and out...
4,mmr,1,95,IGF-CODE-P95-SEN-2,15.8 Regulations for gas detection 15.8.1 Perm...
5,mmr,2,89,IGF-CODE-P89-SEN-2,.2 Operation of the overpressure ventilation s...
6,mmr,3,90,IGF-CODE-P90-SEN-4,13.6.3 Ventilation systems for fuel preparatio...
7,mmr,4,78,IGF-CODE-P78-SEN-1,Suitable alarms shall be provided to indicate ...


### Step 4: Build LangChain-style grounded QA function

Now we connect **retrieval** to **generation** in a single helper function.

#### What `lc_answer_question(...)` is doing (line by line conceptually)

1. **Choose a retriever** based on `strategy` (`similarity` vs `mmr`) and `top_k`.
2. **`retriever.invoke(question)`**: LangChain returns a list of `Document` objects ranked for that question.
3. **`lc_join_context(...)`**: converts retrieved docs into one big context string with `[i] (p.X)` markers so the model (and humans) can refer to evidence positions.

#### Why we still call `call_local_llm(...)` instead of a LangChain chat model here

For teaching, we want a controlled experiment:

- **Change**: retrieval orchestration (LangChain retriever mode).
- **Keep constant**: prompt format and local model backend.

So we reuse `build_grounded_prompt(...)` when it exists, then call the same `call_local_llm(...)` you used in Section A. That makes differences in answers easier to attribute to retrieval behavior.

#### What you should look for in the printed answer preview

- Does the answer cite evidence in a grounded way (as your prompt requires)?
- Does MMR produce answers that mention **broader** aspects of the code vs similarity-only retrieval?

If `call_local_llm` is not defined yet, go back to Section A LLM setup cells before running this step.

In [55]:
def lc_join_context(docs):
    blocks = []
    for i, d in enumerate(docs, 1):
        page = d.metadata.get('page', '?')
        blocks.append(f'[{i}] (p.{page}) {d.page_content}')
    return '\n\n'.join(blocks)


def lc_answer_question(question: str, strategy: str = 'mmr', top_k: int = 4, temperature: float = 0.2):
    if strategy == 'similarity':
        retriever = lc_vectorstore.as_retriever(search_type='similarity', search_kwargs={'k': top_k})
    elif strategy == 'mmr':
        retriever = lc_vectorstore.as_retriever(search_type='mmr', search_kwargs={'k': top_k, 'fetch_k': max(12, top_k * 3), 'lambda_mult': 0.5})
    else:
        raise ValueError("strategy must be 'similarity' or 'mmr'")

    docs = retriever.invoke(question)
    context = lc_join_context(docs)

    if 'build_grounded_prompt' in globals():
        prompt = build_grounded_prompt(question, context)
    else:
        prompt = f"Question: {question}\n\nContext:\n{context}\n\nAnswer with citations."

    answer = call_local_llm(prompt, max_tokens=320, temperature=temperature)

    return {
        'question': question,
        'strategy': strategy,
        'retrieved_docs': docs,
        'context': context,
        'answer': answer,
    }

lc_result = lc_answer_question(lc_demo_query, strategy='mmr', top_k=4)
print('Question:', lc_result['question'])
print('Strategy:', lc_result['strategy'])
print('\nAnswer preview:\n')
print(lc_result['answer'])

llama_context: n_ctx_seq (4096) > n_ctx_train (2048) -- possible training context overflow


Question: For IGF compliance, what should operators verify for ventilation and gas detection systems?
Strategy: mmr

Answer preview:

Example:
Final Answer:
Evidence Used:
1) Operators verify for ventilation and gas detection systems
2) Operators verify for ventilation and gas detection systems
3) Confidence: Medium


### Step 5: Compare retrieval strategies across questions

This step is a **mini evaluation loop**: the same three questions are answered twice (similarity vs MMR).

#### How to interpret the output table

Each row corresponds to one `(question, strategy)` pair. Columns are designed for quick classroom discussion:

- **`retrieved_pages`**: which PDF pages showed up in the top-k evidence. If MMR is working as intended, you sometimes see **more spread** across pages compared to pure similarity (not always; it depends on the query and chunking).
- **`answer_len`**: rough proxy for verbosity (not quality by itself).
- **`answer_preview`**: a short snippet for fast scanning.

#### What “good” looks like (practical guidance)

Good RAG answers are not necessarily the longest. Look for:

- grounded statements tied to retrieved pages,
- fewer contradictions,
- less “copy-paste repetition” from a single passage.

#### How to extend this exercise

- Add your own questions in `lc_questions`.
- Try `top_k=6` vs `top_k=3` and observe evidence vs hallucination risk.
- Change temperature in `lc_answer_question(...)` calls (if you want to study stability).

This ends Section C: you should now be able to narrate the full path **chunks → LangChain vector store → retriever → prompt → local LLM**, and explain why MMR might change the evidence bundle.

In [56]:
lc_questions = [
    'Summarize key safety obligations in the IGF Code.',
    'What requirements are stated for fire protection systems?',
    'How does the code discuss emergency preparedness?'
]

lc_rows = []
for q in lc_questions:
    for strategy in ['similarity', 'mmr']:
        try:
            r = lc_answer_question(q, strategy=strategy, top_k=4, temperature=0.2)
            pages = [d.metadata.get('page') for d in r['retrieved_docs']]
            lc_rows.append({
                'question': q,
                'strategy': strategy,
                'retrieved_pages': sorted({p for p in pages if p is not None}),
                'answer_len': len(r['answer']),
                'answer_preview': r['answer'][:220].replace('\n', ' ')
            })
        except Exception as e:
            lc_rows.append({
                'question': q,
                'strategy': strategy,
                'retrieved_pages': [],
                'answer_len': 0,
                'answer_preview': f'error: {e}'
            })

lc_compare_df = pd.DataFrame(lc_rows)
display(lc_compare_df)

llama_context: n_ctx_seq (4096) > n_ctx_train (2048) -- possible training context overflow
llama_context: n_ctx_seq (4096) > n_ctx_train (2048) -- possible training context overflow
llama_context: n_ctx_seq (4096) > n_ctx_train (2048) -- possible training context overflow
llama_context: n_ctx_seq (4096) > n_ctx_train (2048) -- possible training context overflow
llama_context: n_ctx_seq (4096) > n_ctx_train (2048) -- possible training context overflow
llama_context: n_ctx_seq (4096) > n_ctx_train (2048) -- possible training context overflow


,question,strategy,retrieved_pages,answer_len,answer_preview
0,Summarize key safety obligations in the IGF Code.,similarity,"[1, 2, 7, 13]",489,4) Reasoning Example: The IGF Code requires t...
1,Summarize key safety obligations in the IGF Code.,mmr,"[1, 13, 54, 92]",1284,4) Reasoning Final Answer: 1) The IGF Code re...
2,What requirements are stated for fire protecti...,similarity,"[12, 83]",674,Example: 1) Final Answer: The goal of this cha...
3,What requirements are stated for fire protecti...,mmr,"[12, 21, 83, 85]",1292,Example: Final Answer: The goal of this chapte...
4,How does the code discuss emergency preparedness?,similarity,"[96, 120, 121]",1445,Evidence Use: 1) The code discusses emergency ...
5,How does the code discuss emergency preparedness?,mmr,"[8, 96, 119, 120]",947,Final Answer: 1) Final Answer: Emergency prepa...


---

## Section D - Advanced RAG in LlamaIndex

### Why this section exists (for beginners)

**LlamaIndex** is a framework focused on **indexing** and **querying** knowledge with LLMs. Like LangChain, it does not “replace” the fundamentals you learned in Section A; it gives you higher-level objects (documents, indexes, retrievers, query engines) so you can iterate faster on retrieval-heavy workflows.

This section uses the **same** `manual_chunks` as Section A and Section C so you can compare three styles:

1. **Section A**: explicit Python + FAISS + your own functions.
2. **Section C**: LangChain document/vectorstore/retriever APIs.
3. **Section D**: LlamaIndex document/index/retriever APIs.

### Key LlamaIndex vocabulary (read once)

- **`Document`**: one chunk of text plus metadata (similar idea to LangChain’s `Document`, but this is LlamaIndex’s class).
- **`VectorStoreIndex`**: builds a vector index from documents using the configured embedding model (`Settings.embed_model`).
- **`Retriever`**: returns ranked **nodes** (retrieved passages) for a query.
- **Node vs Document (intuition)**: in many LlamaIndex flows, documents become internal “nodes”. For beginners, treat the retrieved `node.text` as the chunk text your model will read.

### Prerequisites

Same as Section C:

- `manual_chunks` must exist from Section A.
- We reuse `embed_model_name` when available so embeddings stay aligned with the rest of the notebook.
- For generation, `call_local_llm(...)` should exist from Section A.

### What you will practice here

- Configure a global embedding model via `Settings.embed_model`.
- Build `VectorStoreIndex.from_documents(...)`.
- Inspect retrieval results in a table before calling the LLM.
- Run a small multi-question evaluation snapshot.

### Learning goal

By the end of Section D you should be able to explain: *What does LlamaIndex optimize for compared to writing raw FAISS code?* and *What information should you always print when debugging retrieval (pages, scores, previews)?*

### Step 1: Install LlamaIndex packages

We install two pieces:

- **`llama-index`**: the core library (documents, indexing, retrieval, query engines, etc.).
- **`llama-index-embeddings-huggingface`**: integration that exposes HuggingFace/Sentence-Transformers embeddings through LlamaIndex’s embedding interface.

#### Why a separate embeddings package?

LlamaIndex keeps optional integrations modular so installs stay smaller until you need them. Here we explicitly need HuggingFace embeddings to match the rest of this notebook’s embedding approach.

#### After installation

If imports fail immediately after pip, restart the kernel once, rerun the install cell, then rerun the import cell.

Next step wires `Settings.embed_model` so every index operation uses the same embedding model name you chose earlier (`embed_model_name` when present).

In [57]:
%pip install -q llama-index llama-index-embeddings-huggingface

Note: you may need to restart the kernel to use updated packages.


### Step 2: Build LlamaIndex documents and vector index

#### What the code cell is doing

1. **`LIDocument(text=..., metadata=...)`**: converts each `manual_chunks` item into a LlamaIndex document. Metadata is crucial for transparency (page numbers, chunk ids).
2. **`Settings.embed_model = HuggingFaceEmbedding(...)`**: sets the **global default** embedding model for subsequent LlamaIndex operations in this notebook session.
3. **`VectorStoreIndex.from_documents(li_docs)`**: embeds all documents and constructs a vector index you can query.

#### Relationship to Section A and Section C

- Section A: you manually managed vectors and FAISS.
- Section C: LangChain built its own FAISS wrapper.
- Section D: LlamaIndex builds its own internal vector store/index.

All three can coexist; they are different “front doors” to the same underlying idea: **embedding + nearest neighbor search**.

#### Performance note for beginners

This step embeds the corpus again. Expect:

- first-time model download latency,
- CPU/GPU time depending on your machine.

#### Expected output

You should see counts for documents and the embedding model name. If `manual_chunks` is missing, return to Section A chunking.

In [58]:
from llama_index.core import Document as LIDocument
from llama_index.core import Settings, VectorStoreIndex
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

if 'manual_chunks' not in globals() or not manual_chunks:
    raise RuntimeError('Please run Section A up to chunking step to create manual_chunks.')

li_docs = [
    LIDocument(
        text=c['text'],
        metadata={
            'chunk_id': c.get('chunk_id'),
            'page': c.get('page'),
            'chunk_method': c.get('chunk_method')
        }
    )
    for c in manual_chunks
]

li_embedding_model_name = globals().get('embed_model_name', 'sentence-transformers/all-MiniLM-L6-v2')
Settings.embed_model = HuggingFaceEmbedding(model_name=li_embedding_model_name)

li_index = VectorStoreIndex.from_documents(li_docs)
print('LlamaIndex docs:', len(li_docs))
print('Embedding model:', li_embedding_model_name)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6829.71it/s]


LlamaIndex docs: 604
Embedding model: sentence-transformers/all-MiniLM-L6-v2


### Step 3: Configure retriever and inspect retrieved nodes

This step intentionally does **retrieval only** (no LLM call yet). In real projects, this is one of the highest ROI debugging habits: *verify retrieval before blaming the model.*

#### What `as_retriever(similarity_top_k=5)` means

`similarity_top_k` is LlamaIndex’s way of saying: “return the top 5 most similar passages to the query embedding.”

It is conceptually the same knob as `k` in LangChain similarity search, but the parameter name differs by framework.

#### How to read the preview table

- **`rank`**: 1 is the retriever’s top choice.
- **`score`**: if present, it is a similarity-related score from the retriever/node object (exact interpretation can vary by version/backend). Use it as a **relative** signal within one query, not an absolute physical unit.
- **`page` / `chunk_id`**: traceability back to your chunking output.
- **`text_preview`**: quick sanity check that the retrieved text is on-topic.

#### Debugging prompts for learners

Ask yourself:

- Do the top passages actually mention the query topic?
- Are you retrieving boilerplate / headers repeatedly?
- If retrieval is bad, would better chunking or a better embedding model help more than changing the LLM?

Once retrieval looks reasonable, proceed to Step 4 to add generation.

In [59]:
li_retriever = li_index.as_retriever(similarity_top_k=5)
li_demo_query = globals().get('user_question', 'What does IGF Code require for fire safety?')
li_nodes = li_retriever.retrieve(li_demo_query)

li_preview_rows = []
for rank, node in enumerate(li_nodes, 1):
    li_preview_rows.append({
        'rank': rank,
        'score': getattr(node, 'score', None),
        'page': node.metadata.get('page') if hasattr(node, 'metadata') else None,
        'chunk_id': node.metadata.get('chunk_id') if hasattr(node, 'metadata') else None,
        'text_preview': node.text[:180].replace('\n', ' ')
    })

display(pd.DataFrame(li_preview_rows))

,rank,score,page,chunk_id,text_preview
0,1,0.588124,96,IGF-CODE-P96-SEN-1,15.8.8 Audible and visible alarms from the gas...
1,2,0.583514,95,IGF-CODE-P95-SEN-1,15.7 Regulations for gas engine monitoring In ...
2,3,0.578240,90,IGF-CODE-P90-SEN-4,13.6.3 Ventilation systems for fuel preparatio...
3,4,0.561961,90,IGF-CODE-P90-SEN-0,"MSC 95/22/Add.1 Annex 1, page 90 I:\MSC\95\MSC..."
4,5,0.560283,89,IGF-CODE-P89-SEN-4,13.4 Regulations for tank connection space 13....


### Step 4: Build LlamaIndex QA function with notebook LLM backend

LlamaIndex can orchestrate the full query engine (retriever + LLM + prompt templates). In this workshop notebook we use a **deliberately simplified** pattern:

1. Use LlamaIndex retrieval (`retrieve(...)`) to select evidence nodes.
2. Join node texts into a single context string (`li_join_context`).
3. Reuse your grounded prompt builder (`build_grounded_prompt`) when available.
4. Call **`call_local_llm(...)`** for generation, same as Section A.

#### Why this design?

Beginners often confuse “framework magic” with “model intelligence.” By keeping the generator fixed, you isolate what LlamaIndex changed: **how evidence is selected and packaged**.

#### What `li_answer_question(...)` returns

A small Python dict containing:

- `nodes`: raw retrieved objects (useful for deeper debugging),
- `context`: the exact string passed into the prompt,
- `answer`: model output.

#### What to compare against Section C

For the same question, compare:

- retrieved pages,
- context packaging,
- answer style and grounding.

If answers differ wildly but retrieved pages are similar, the issue may be prompt/temperature/max tokens—not the framework.

If retrieved pages differ, retrieval configuration is the first knob to tune (`similarity_top_k`).

In [60]:
def li_join_context(nodes):
    blocks = []
    for i, n in enumerate(nodes, 1):
        page = n.metadata.get('page', '?') if hasattr(n, 'metadata') else '?'
        blocks.append(f'[{i}] (p.{page}) {n.text}')
    return '\n\n'.join(blocks)


def li_answer_question(question: str, top_k: int = 5, temperature: float = 0.2):
    retriever = li_index.as_retriever(similarity_top_k=top_k)
    nodes = retriever.retrieve(question)
    context = li_join_context(nodes)

    if 'build_grounded_prompt' in globals():
        prompt = build_grounded_prompt(question, context)
    else:
        prompt = f"Question: {question}\n\nContext:\n{context}\n\nAnswer with citations."

    answer = call_local_llm(prompt, max_tokens=320, temperature=temperature)

    return {
        'question': question,
        'nodes': nodes,
        'context': context,
        'answer': answer,
    }

li_result = li_answer_question(li_demo_query, top_k=5)
print('Question:', li_result['question'])
print('\nAnswer preview:\n')
print(li_result['answer'][:1200])

llama_context: n_ctx_seq (4096) > n_ctx_train (2048) -- possible training context overflow


Question: For IGF compliance, what should operators verify for ventilation and gas detection systems?

Answer preview:

Evidence:
1) The retrieved context:
[1] (p.96) 15.8.8 Audible and visible alarm s from the gas detection equipment shall be located on the navigation bridge or in the continuously manned central control station.

2) The retrieved context:
[2] (p.95) 15.7 Regulations for gas engine monitoring In addition to the instrumentation provided in accordance with part C of SOLAS chapter II-1, indicator s shall be fitted on the navigation bridge, the engine control room and the manoeuvring platform for: .1 operation of the engine in case of gas-only engines; or .2 operation and mode of operation of the engine in the case of dual fuel engines.

3) The retrieved context:
[3] (p.90) 13.6.3 Ventilation systems for fuel preparation rooms, shall be in operation when pump s or compressors are working.

4) The retrieved context:
[4] (p.90) MSC 95/22/Add.1 Annex 1, page 90 I:\MSC\95\MSC 

### Step 5: Multi-question evaluation snapshot

This is a lightweight “benchmark sheet” for three fixed questions.

#### How to read the evaluation table

Each row is one question run through `li_answer_question(...)`.

- **`retrieved_pages`**: quick proxy for evidence breadth. If you always see the same narrow page range, your queries may be too vague, your `top_k` too small, or your chunks may be dominated by repeated template text.
- **`answer_len`**: only a rough signal; pair it with manual reading of full answers when grading quality.
- **`answer_preview`**: fast scan for hallucination red flags (specific numbers with no support, unrelated regulations, etc.).

#### Suggested classroom activity

1. Pick one question where retrieval looks weak: improve **query wording** first (add constraints like “list requirements”, “cite page numbers”).
2. Increase `similarity_top_k` slightly and observe whether evidence improves or becomes noisy.
3. Compare the best LlamaIndex run to your Section A baseline answer for the same question.

#### Where to go next (optional self-study)

LlamaIndex supports richer composition (query engines, response synthesizers, rerankers, hybrid retrieval). This notebook keeps scope small so beginners finish with a clear mental model first.

You have now completed Section D: **documents → Settings embed model → vector index → retriever → grounded prompt → local LLM**, with explicit tables for transparency.

In [61]:
li_questions = [
    'Summarize key compliance themes in the IGF Code.',
    'What are key fire safety provisions mentioned in the code?',
    'What guidance appears related to emergency procedures?'
]

li_rows = []
for q in li_questions:
    try:
        r = li_answer_question(q, top_k=5, temperature=0.2)
        pages = []
        for n in r['nodes']:
            if hasattr(n, 'metadata'):
                pages.append(n.metadata.get('page'))
        li_rows.append({
            'question': q,
            'retrieved_pages': sorted({p for p in pages if p is not None}),
            'answer_len': len(r['answer']),
            'answer_preview': r['answer'][:220].replace('\n', ' ')
        })
    except Exception as e:
        li_rows.append({
            'question': q,
            'retrieved_pages': [],
            'answer_len': 0,
            'answer_preview': f'error: {e}'
        })

li_eval_df = pd.DataFrame(li_rows)
display(li_eval_df)

llama_context: n_ctx_seq (4096) > n_ctx_train (2048) -- possible training context overflow
llama_context: n_ctx_seq (4096) > n_ctx_train (2048) -- possible training context overflow
llama_context: n_ctx_seq (4096) > n_ctx_train (2048) -- possible training context overflow


,question,retrieved_pages,answer_len,answer_preview
0,Summarize key compliance themes in the IGF Code.,"[7, 13, 32, 91]",84,4) Recommendations (if any) 5) References (if ...
1,What are key fire safety provisions mentioned ...,"[11, 12, 83, 85, 92]",1251,Example: Final Answer: 1) Final Answer: Eviden...
2,What guidance appears related to emergency pro...,"[13, 89, 96, 119]",0,
